# Photothermal Parameter Estimation: PINN vs Classical Methods

This notebook compares three methods for recovering the **heating rate** $a$ and **thermal time constant** $\tau$ from laser-induced $T(t)$ data of gold nanoparticle (GNP) suspensions:

1. **Roper** — log-linear fit to the cooling tail
2. **Whole-curve least squares** — global fit to the analytic ODE solution
3. **Physics-Informed Neural Network (PINN)** — data loss + embedded ODE residual

Experiments cover synthetic data (controlled ground truth) and five real heating–cooling curves digitised from three open-access papers.

In [ ]:
import os
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use('Agg')  # headless mode for scripts

import functools
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import seaborn as sns
import pandas as pd
from scipy.optimize import curve_fit

from network import Net, NetDiscovery, np_to_th, DEVICE
from diff_equations import grad, cooling_law

sns.set_theme()
torch.manual_seed(42)
torch.set_num_threads(4)

---
## 2 · Physical Model

### Governing ODE

$$\frac{dT}{dt} = a\,L(t) - \frac{T - T_{\mathrm{env}}}{\tau}$$

where $L(t) = 1$ while the laser is on ($t < t_{\mathrm{off}}$) and $L(t) = 0$ afterwards, $a$ (°C/s) is the effective heating rate, and $\tau$ (s) is the thermal time constant. Throughout, $\theta = T - T_{\mathrm{env}}$ denotes the temperature rise above ambient.

### Analytic solution

$$T(t) = T_{\mathrm{env}} + \begin{cases}a\tau\!\left(1 - e^{-t/\tau}\right) & t < t_{\mathrm{off}} \\a\tau\!\left(1 - e^{-t_{\mathrm{off}}/\tau}\right)e^{-(t-t_{\mathrm{off}})/\tau} & t \ge t_{\mathrm{off}}\end{cases}$$

### Roper bias

The Roper method estimates $\hat\tau$ from the cooling tail and sets $\hat a = \Delta T_{\max}/\hat\tau$, which assumes $\Delta T_{\max} = a\tau$ (steady state). Because $\Delta T_{\max} = a\tau\,(1 - e^{-t_{\mathrm{off}}/\tau})$,

$$\frac{\hat a}{a} = \frac{\tau}{\hat\tau}\,\bigl(1 - e^{-t_{\mathrm{off}}/\tau}\bigr).$$

The second factor is the heating-truncation bias. With an exact cooling fit ($\hat\tau = \tau$) the bias is $-e^{-t_{\mathrm{off}}/\tau}$: $\approx -37\,\%$ at $t_{\mathrm{off}}/\tau = 1$ and $\approx -0.7\,\%$ at $t_{\mathrm{off}}/\tau = 5$. The first factor departs from one when the cooling phase alone does not follow the relaxation time of the complete trace, which is what happens on the real curves of Section 10.

---
## 3 · Analytic Solution and Synthetic Data

**`heating_cooling(t, a, tau, t_off, Tenv)`** — exact solution of $d\theta/dt = a\,L(t) - \theta/\tau$ (Section 2)

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | times at which the temperature is evaluated (array) |
| `a` | $a$ | °C/s | heating rate |
| `tau` | $\tau$ | s | thermal time constant |
| `t_off` | $t_{\mathrm{off}}$ | s | laser switch-off time |
| `Tenv` | $T_{\mathrm{env}}$ | °C | ambient temperature (default 25) |

**Returns:** the temperature $T(t)$ in °C.

In [ ]:
def heating_cooling(t, a, tau, t_off, Tenv=25.0):
    """Exact analytic solution of the photothermal ODE."""
    theta_off = a * tau * (1 - np.exp(-t_off / tau))
    return Tenv + np.where(
        t < t_off,
        a * tau * (1 - np.exp(-t / tau)),
        theta_off * np.exp(-(t - t_off) / tau),
    )

In [ ]:
# True parameter values (to be recovered by each method)
np.random.seed(1)
Tenv_true  = 25.0    # ambient temperature [°C]
a_true     = 0.05    # heating rate [°C/s]
tau_true   = 300.0   # thermal time constant [s]
t_off_true = 1 * tau_true # laser switch-off time [s]  (short experiment: t_off/τ = 1)

times = np.arange(0, 3 * tau_true, 5)
eq    = functools.partial(heating_cooling, a=a_true, tau=tau_true, t_off=t_off_true, Tenv=Tenv_true)
temps = eq(times)

# Training data: 10 s sampling interval, 0.1 °C Gaussian noise
t = np.arange(0, 3 * tau_true + 5, 10)
T = eq(t) + 0.1 * np.random.randn(len(t))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(times / 60, temps, lw=2, label='Exact solution')
ax.plot(t / 60, T, 'o', ms=3, alpha=0.6, label='Simulated data (0.1 °C noise)')
ax.axvline(t_off_true / 60, color='gray', ls='--', lw=0.8, label='laser off')
ax.set_xlabel('Time (min)'); ax.set_ylabel('Temperature (°C)')
ax.set_title(f'Synthetic experiment — $t_{{\\mathrm{{off}}}}/\\tau = {t_off_true/tau_true:.0f}$')
ax.legend()
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig0_synthetic_data.jpg', dpi=150, bbox_inches='tight')
plt.show()


---
## 4 · Classical Baselines

### 4.1 Roper method

Estimates $\tau$ from the log-linear slope of the cooling phase, then computes
$\hat{a} = \Delta T_{\max} / \hat{\tau}$.

### 4.2 Whole-curve least-squares

Fits the exact analytic solution (heating + cooling) to all data points via `scipy.optimize.curve_fit`, with $T_{\mathrm{env}}$ fixed.

**`roper(t, T, t_off, Tenv_, min_frac)`** — Roper method: $\ln\theta$ is fitted on the cooling tail to get $\hat\tau$, then $\hat a = \Delta T_{\max}/\hat\tau$ (Section 4)

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | measured times (array) |
| `T` | $T$ | °C | measured temperatures (array) |
| `t_off` | $t_{\mathrm{off}}$ | s | laser switch-off time |
| `Tenv_` | $T_{\mathrm{env}}$ | °C | ambient temperature, subtracted to get $\theta = T - T_{\mathrm{env}}$ |
| `min_frac` | — | — | cooling points are used while $\theta > \texttt{min\_frac}\cdot\Delta T_{\max}$ (default 0.05) |

**Returns:** `(a_est, tau_est)` in °C/s and s.

In [ ]:
def roper(t, T, t_off, Tenv_=Tenv_true, min_frac=0.05):
    """Standard Roper et al. 2007 method."""
    theta  = T - Tenv_
    dT_max = theta[t <= t_off].max()
    cool   = (t > t_off) & (theta > min_frac * dT_max)
    slope, _ = np.polyfit(t[cool] - t_off, np.log(theta[cool]), 1)
    tau_est  = -1.0 / slope
    return dT_max / tau_est, tau_est

**`whole_curve_fit(t, T, t_off, Tenv_)`** — least-squares fit of `heating_cooling` to all points, $\min_{a,\tau}\sum_i (T_i - T_{\mathrm{model}}(t_i; a, \tau))^2$ (Section 4)

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | measured times (array) |
| `T` | $T$ | °C | measured temperatures (array) |
| `t_off` | $t_{\mathrm{off}}$ | s | laser switch-off time (fixed, not fitted) |
| `Tenv_` | $T_{\mathrm{env}}$ | °C | ambient temperature (fixed, not fitted) |

**Returns:** `(a_est, tau_est)` in °C/s and s. Fit bounds: $a \in [10^{-6}, 10]$, $\tau \in [1, 10^5]$; start values $a = 0.01$, $\tau = 100$.

In [ ]:
def whole_curve_fit(t, T, t_off, Tenv_=Tenv_true):
    """Least-squares fit to the full analytic ODE solution."""
    f = lambda t_, a_, tau_: heating_cooling(t_, a_, tau_, t_off, Tenv_)
    (a_est, tau_est), _ = curve_fit(f, t, T, p0=(0.01, 100.0),
                                    bounds=([1e-6, 1.0], [10.0, 1e5]))
    return a_est, tau_est

In [ ]:
a_r,  tau_r  = roper(t, T, t_off_true)
a_wc, tau_wc = whole_curve_fit(t, T, t_off_true)

print(f"True values :  a = {a_true:.4f} °C/s    τ = {tau_true:.0f} s")
print(f"Roper       :  a = {a_r:.4f} °C/s  ({100*(a_r-a_true)/a_true:+.1f} %)    τ = {tau_r:.0f} s")
print(f"Whole-curve :  a = {a_wc:.4f} °C/s  ({100*(a_wc-a_true)/a_true:+.1f} %)    τ = {tau_wc:.0f} s")

---
## 5 · Physics-Informed Neural Network (PINN)

### 5.1 Architecture and scaling

The network is a 4-hidden-layer MLP (100 units each, **tanh** activations). For stable training, time and temperature rise are scaled to about $[0, 1]$:

$$\hat t = \frac{t}{t_{\mathrm{scale}}}, \qquad \hat\theta = \frac{\theta}{T_{\mathrm{scale}}}, \qquad t_{\mathrm{scale}} = \max t, \quad T_{\mathrm{scale}} = \max\theta,$$

and the network maps the input

$$\mathbf{x}(\hat t) = \bigl[\,\hat t,\;\mathrm{ReLU}(\hat t - \hat t_{\mathrm{off}})\,\bigr], \qquad \hat t_{\mathrm{off}} = t_{\mathrm{off}}/t_{\mathrm{scale}},$$

to the scaled temperature rise $\hat\theta(\hat t)$. The second input channel encodes the laser switch-off corner, which a smooth network struggles to represent with a single input (Section 8).

Two scalar parameters are learned alongside the weights; both are in scaled units and are converted back after training:

| Symbol | Code | Meaning | Back to physical units |
|---|---|---|---|
| $\hat a$ | `model.a` | scaled heating rate | $a = \hat a\,T_{\mathrm{scale}}/t_{\mathrm{scale}}$ |
| $\hat\kappa$ | `model.kappa` | scaled loss rate (the linear loss $\theta/\tau$ in scaled units) | $\tau = t_{\mathrm{scale}}/\hat\kappa$ |

The scaled ODE is $d\hat\theta/d\hat t = \hat a\,L - \hat\kappa\,\hat\theta$. Note that $\hat\kappa$ exists only for the linear loss; the effective loss rate $r(\theta) = f(\theta)/\theta$ of Section 11 is a different, unscaled quantity.

### 5.2 Loss function

$$\mathcal{L} = \underbrace{\frac{1}{N}\sum_i \bigl(\hat\theta(\hat t_i) - \hat\theta^{\mathrm{obs}}_i\bigr)^2}_{\text{data}} + \lambda\biggl[\underbrace{\frac{1}{M}\sum_j \left(\hat a\,L(\hat t_j) - \hat\kappa\,\hat\theta(\hat t_j) - \frac{d\hat\theta}{d\hat t}\bigl|_{\hat t_j}\right)^2}_{\text{physics (ODE residual)}} + \underbrace{\hat\theta(0)^2}_{\text{initial condition}}\biggr]$$

where $\hat\theta^{\mathrm{obs}}_i$ are the scaled observations, the $M = 800$ collocation times $\hat t_j$ are spread evenly over $[0, 1]$ (`t_col` in the code), and $\lambda = 1$.

### 5.3 Optimisation

1. **Adam** — 3 000 steps at learning rate $\alpha = 3\times10^{-3}$ (input scaling enables a large learning rate)
2. **L-BFGS** — at most 300 iterations with strong Wolfe line search (final convergence to high precision); PyTorch also stops after 375 function evaluations or when its tolerances are reached, and Section 12.2 reports what was really used

### 5.4 Code names used in the notebook

| Math | Code |
|---|---|
| $t_{\mathrm{scale}},\; T_{\mathrm{scale}}$ | `t_scale`, `T_scale` |
| $\hat t,\; \hat\theta^{\mathrm{obs}}$ (scaled data) | `t_hat`, `theta_hat` |
| $\hat t_{\mathrm{off}}$ | `t_off_hat` |
| collocation times $\hat t_j$ | `t_col` |
| true parameters of the synthetic data | globals `a_true`, `tau_true`, `Tenv_true`, `t_off_true` |
| fitted values | `a_est`, `tau_est`, or `a_`, `tau_` inside loops |

A trailing underscore (`t_off_`, `Tenv_`, `T_`) only marks a function argument that would otherwise shadow one of the globals.

**`PhotothermalPINN(t_off_hat, switch_input, ...)`** — network for the linear loss (Section 5): $\hat t \mapsto \hat\theta$, with learnable $\hat a$ (`a`) and $\hat\kappa$ (`kappa`)

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t_off_hat` | $\hat t_{\mathrm{off}}$ | — | scaled switch-off time, $t_{\mathrm{off}}/t_{\mathrm{scale}}$ |
| `switch_input` | — | — | `True`: inputs $[\hat t, \mathrm{ReLU}(\hat t - \hat t_{\mathrm{off}})]$; `False`: only $\hat t$ |
| `n_units` | — | — | neurons per hidden layer (default 100; 4 hidden layers, tanh) |
| `epochs` | — | — | Adam steps in `fit` (default 3000) |
| `lr` | $\alpha$ | — | Adam learning rate (default $3\times10^{-3}$) |
| `data_loss` | — | — | data loss (default MSE) |
| `physics_loss` | — | — | physics loss `physics_loss(model)` (ODE residual + initial condition) |
| `data_weight`, `physics_weight` | $w_{\mathrm{data}}$, $\lambda$ | — | weights of the data loss and of the physics loss in `loss = data_weight × data_loss + physics_weight × physics_loss` (both 1 in all runs) |

**Forward:** takes $\hat t$ of shape (N, 1) and returns $\hat\theta$ of shape (N, 1).

In [ ]:
class PhotothermalPINN(NetDiscovery):
    """PINN for discovering (a, τ) from photothermal heating–cooling data."""

    def __init__(self, t_off_hat, switch_input=True,
                 n_units=100, epochs=3000, data_loss=nn.MSELoss(),
                 lr=3e-3, physics_loss=None, data_weight=1.0, physics_weight=1.0):
        n_in = 2 if switch_input else 1
        super().__init__(input_dim=n_in, output_dim=1,
                         n_units=n_units, epochs=epochs,
                         data_loss=data_loss, lr=lr,
                         physics_loss=physics_loss,
                         data_weight=data_weight, physics_weight=physics_weight)

        # Replace ReLU activations with tanh for smooth ODE gradients
        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.ReLU):
                self.layers[i] = nn.Tanh()

        self.t_off_hat    = t_off_hat
        self.switch_input = switch_input
        self.a            = nn.Parameter(data=torch.tensor([0.]))

    def forward(self, t):
        if self.switch_input:
            t = torch.cat([t, torch.relu(t - self.t_off_hat)], dim=1)
        return self.out(self.layers(t))

**`train_photothermal(t_, T_, t_off_, Tenv_, switch_input, data_weight, physics_weight, adam_epochs, lbfgs_iters, seed, return_fit, log)`** — scales the data, trains `PhotothermalPINN` with the loss of Section 5.2, converts back with $a = \hat a\,T_{\mathrm{scale}}/t_{\mathrm{scale}}$, $\tau = t_{\mathrm{scale}}/\hat\kappa$

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t_` | $t$ | s | measured times (array) |
| `T_` | $T$ | °C | measured temperatures (array) |
| `t_off_` | $t_{\mathrm{off}}$ | s | laser switch-off time |
| `Tenv_` | $T_{\mathrm{env}}$ | °C | ambient temperature |
| `switch_input` | — | — | use the switch input (default `True`) |
| `data_weight` | $w_{\mathrm{data}}$ | — | weight of the data loss (default 1) |
| `physics_weight` | $\lambda$ | — | weight of the physics loss, ODE residual + initial condition (default 1) |
| `adam_epochs`, `lbfgs_iters` | — | — | Adam steps (3000) and the ceiling on L-BFGS iterations (300) |
| `seed` | — | — | random seed for the weights |
| `return_fit` | — | — | also return the network's fitted temperature at the data times (default `False`) |
| `log` | — | — | optional `TrainLog` (Section 12) that records the losses, the gradients and the cost of the run (default `None`) |

**Returns:** `(a_est, tau_est)` in °C/s and s, or `(a_est, tau_est, fit)` with `fit` in °C when `return_fit=True`. The physics residual uses $M = 800$ collocation times `t_col` in $[0, 1]$.

In [ ]:
def train_photothermal(t_, T_, t_off_, Tenv_=Tenv_true,
                       switch_input=True, data_weight=1.0, physics_weight=1.0,
                       adam_epochs=3000, lbfgs_iters=300, seed=0, return_fit=False, log=None):
    """Train PhotothermalPINN and return (a_est, tau_est), plus the network's fitted T if return_fit."""
    torch.manual_seed(seed)

    # Normalise inputs to [0, 1]
    t_scale = float(t_.max())
    T_scale = float(max((T_ - Tenv_).max(), 1e-3))
    t_hat, theta_hat = t_ / t_scale, (T_ - Tenv_) / T_scale
    t_off_hat    = t_off_ / t_scale

    def physics_loss(model):
        t_col = torch.linspace(0, 1, 800).view(-1, 1).requires_grad_(True).to(DEVICE)
        laser = (t_col < t_off_hat).float()
        temps = model(t_col)
        dT    = grad(temps, t_col)[0]
        pde   = model.a * laser - model.kappa * temps - dT
        ic    = model(torch.zeros(1, 1).to(DEVICE)).pow(2).mean()
        return torch.mean(pde ** 2) + ic

    model = PhotothermalPINN(t_off_hat, switch_input=switch_input,
                              physics_loss=physics_loss,
                              data_weight=data_weight, physics_weight=physics_weight,
                              epochs=adam_epochs, lr=3e-3).to(DEVICE)   # same device as the data
    if log is not None:
        log.start(model, T_scale / t_scale)
    model.fit(t_hat.reshape(-1, 1), theta_hat.reshape(-1, 1),
              on_step=None if log is None else lambda ep, d, p: log.record("Adam", d, p))
    if log is not None:
        log.end_adam()

    # L-BFGS fine-tuning
    Xt_ = np_to_th(t_hat.reshape(-1, 1))
    yt_ = np_to_th(theta_hat.reshape(-1, 1))
    opt = torch.optim.LBFGS(model.parameters(), lr=1.0,
                              max_iter=lbfgs_iters, line_search_fn='strong_wolfe')
    def _closure():
        opt.zero_grad()
        data_term = model.data_loss(yt_, model(Xt_))
        physics_term = physics_loss(model)
        l = data_weight * data_term + physics_weight * physics_term
        l.backward()
        if log is not None:
            log.record("L-BFGS", data_term, physics_term)
        return l
    opt.step(_closure)
    if log is not None:
        log.finish(opt)

    a_est, tau_est = model.a.item() * T_scale / t_scale, t_scale / model.kappa.item()
    if return_fit:   # the network's own output at the data times, in °C
        fit = model(Xt_).detach().cpu().numpy().ravel() * T_scale + Tenv_
        return a_est, tau_est, fit
    return a_est, tau_est

---
## 6 · Sanity Check — Noiseless Full-Duration Experiment

With zero noise and $t_{\mathrm{off}} = 5\tau$, the whole-curve fit and the PINN recover the true parameters to within 0.05 %, while Roper keeps its heating-truncation bias $-e^{-5} \approx -0.7\,\%$.

In [ ]:
t_sc, T_sc = (
    lambda t_e: (t_e, functools.partial(heating_cooling, a=a_true, tau=tau_true, t_off=5*tau_true, Tenv=Tenv_true)(t_e))
)(np.arange(0, 7 * tau_true, 10))

print(f"True:          a = {a_true:.4f}  τ = {tau_true:.0f} s")
for name, fn in [
    ('Roper',           lambda: roper(t_sc, T_sc, 5*tau_true)),
    ('Whole-curve fit', lambda: whole_curve_fit(t_sc, T_sc, 5*tau_true)),
    ('PINN + switch',   lambda: train_photothermal(t_sc, T_sc, 5*tau_true)),
]:
    a_, tau_ = fn()
    print(f"{name:22s}   a = {a_:.4f} ({100*(a_-a_true)/a_true:+.1f}%)   τ = {tau_:.0f} s ({100*(tau_-tau_true)/tau_true:+.1f}%)")

---
## 7 · Experiment 1 — Full vs Short Heating Duration

Same nanoparticle, same noise level (0.1 °C), different laser-on duration.

| Experiment | $t_{\mathrm{off}}$ | $t_{\mathrm{off}}/\tau$ |
|---|---|---|
| Full | $5\tau = 1500$ s | 5 |
| Short | $1\tau = 300$ s | 1 |

**`make_experiment(t_off_e, t_end, dt, noise, seed)`** — synthetic noisy experiment from `heating_cooling` with the true values `a_true`, `tau_true`, `Tenv_true`

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t_off_e` | $t_{\mathrm{off}}$ | s | laser switch-off time |
| `t_end` | — | s | end time of the record |
| `dt` | — | s | sampling interval (default 10) |
| `noise` | — | °C | standard deviation of the Gaussian noise (default 0.1) |
| `seed` | — | — | random seed of the noise |

**Returns:** `(t_e, T_e)` — times and noisy temperatures.

In [ ]:
def make_experiment(t_off_e, t_end, dt=10, noise=0.1, seed=0):
    rng = np.random.default_rng(seed)
    t_e = np.arange(0, t_end + dt / 2, dt)
    T_e = functools.partial(heating_cooling, a=a_true, tau=tau_true, t_off=t_off_e, Tenv=Tenv_true)(t_e)
    return t_e, T_e + rng.normal(0, noise, len(t_e))

EXPERIMENTS = {
    "full  (heat 5τ, cool 5τ)":  dict(t_off_e=5 * tau_true, t_end=10 * tau_true),
    "short (heat 1τ, cool 2τ)": dict(t_off_e=1 * tau_true, t_end= 3 * tau_true),
}

In [ ]:
METHODS = {
    "Roper":           roper,
    "whole-curve fit": whole_curve_fit,
    "PINN (no switch)": lambda t_, T_, to: train_photothermal(t_, T_, to, switch_input=False),
    "PINN + switch":    lambda t_, T_, to: train_photothermal(t_, T_, to, switch_input=True),
}

rows, fits = [], {}
for exp_name, e in EXPERIMENTS.items():
    te, Te = make_experiment(**e, seed=1)
    for m_name, fn in METHODS.items():
        a_, tau_ = fn(te, Te, e['t_off_e'])
        fits[(exp_name, m_name)] = (a_, tau_)
        rows.append(dict(experiment=exp_name, method=m_name,
                         a_err=round(100 * (a_ - a_true) / a_true, 1),
                         tau_err=round(100 * (tau_ - tau_true) / tau_true, 1)))

df1 = pd.DataFrame(rows)
print("=== Error in a [%] ===")
print(df1.pivot(index='method', columns='experiment', values='a_err').to_string())

### Figure 1 — Fitted curves on the short experiment

In [ ]:
e_sh = EXPERIMENTS["short (heat 1τ, cool 2τ)"]
t_sh, T_sh = make_experiment(**e_sh, seed=1)
tt = np.linspace(0, t_sh.max(), 1000)

styles = {
    "Roper":           dict(color='tab:orange', ls='--', lw=1.8),
    "whole-curve fit": dict(color='tab:green',  ls='-.', lw=1.8),
    "PINN (no switch)": dict(color='tab:red',   ls=':',  lw=1.8),
    "PINN + switch":   dict(color='tab:blue',   ls='-',  lw=2.0),
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_sh / 60, T_sh, '.', ms=3, color='lightgray', zorder=1, label='data')
ax.plot(tt / 60, eq(tt), 'k-', lw=1.2, alpha=0.5, label='true')

for m_name, sty in styles.items():
    a_e, tau_e = fits[("short (heat 1τ, cool 2τ)", m_name)]
    err = 100 * (a_e - a_true) / a_true
    ax.plot(tt / 60,
            heating_cooling(tt, a_e, tau_e, e_sh['t_off_e'], Tenv_true),
            label=f'{m_name}  (Δa={err:+.1f}%)', **sty)

ax.axvline(e_sh['t_off_e'] / 60, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Temperature (°C)')
ax.set_title(r'Short experiment ($t_{\mathrm{off}}/\tau = 1$) — method comparison')
ax.legend(fontsize=8)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig1_short_experiment.jpg', dpi=150, bbox_inches='tight')
plt.show()


---
## 8 · Why the Switch Input Matters

A smooth MLP struggles to represent the sharp corner in $T(t)$ at $t_{\mathrm{off}}$ with a single scalar input $t$.  
Adding $\mathrm{ReLU}(t - t_{\mathrm{off}})$ as a second input provides the network with an explicit signal at the kink. Without it, the error in $a$ over five initialisation seeds spans −29.3 % to +10.0 % on the long experiment and −6.3 % to +0.14 % on the short one (Section 8.1); with it, every run stays within 0.7 %.

In [ ]:
zoom_lo = 0.7 * e_sh['t_off_e']
zoom_hi = 1.5 * e_sh['t_off_e']
tt_w    = np.linspace(zoom_lo, zoom_hi, 600)
mask    = (t_sh > zoom_lo) & (t_sh < zoom_hi)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(t_sh[mask] / 60, T_sh[mask], '.', ms=4, color='lightgray', label='data')
ax.plot(tt_w / 60, eq(tt_w), 'k-', lw=1.2, alpha=0.5, label='true')

for key in ['PINN (no switch)', 'PINN + switch']:
    a_e, tau_e = fits[("short (heat 1τ, cool 2τ)", key)]
    ax.plot(tt_w / 60,
            heating_cooling(tt_w, a_e, tau_e, e_sh['t_off_e'], Tenv_true),
            label=key, **styles[key])

ax.axvline(e_sh['t_off_e'] / 60, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Zoom around laser switch-off')
ax.legend(fontsize=9)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig2_switch_zoom.jpg', dpi=150, bbox_inches='tight')
plt.show()


### 8.1 Is the switch effect reproducible?

Section 7 trains each network once. Here the linear-loss PINN is trained five times on the same data (noise seed 1, as in Section 7), with and without the switch input, changing only the initialisation seed. The errors are saved in `results/switch_ablation.csv`.

**Result.** Without the switch input the error in $a$ spans −29.3 % to +10.0 % (long experiment, SD 17.1 %) and −6.3 % to +0.14 % (short, SD 2.8 %); the same seed gives different errors on different hardware, so the unswitched network is not reproducible. With the switch input all ten runs stay within 0.7 % (SD 0.32 % and 0.02 %).

In [ ]:
# Section 8.1: same data as Section 7, five initialisation seeds, with and without the switch input
from contextlib import redirect_stdout
import io

abl_rows = []
for exp_name, e in EXPERIMENTS.items():
    te, Te = make_experiment(**e, seed=1)
    for switch in (False, True):
        for seed_ in range(5):
            with redirect_stdout(io.StringIO()):          # hide the per-epoch log
                a_, tau_ = train_photothermal(te, Te, e['t_off_e'], switch_input=switch, seed=seed_)
            abl_rows.append(dict(experiment=exp_name, switch=switch, seed=seed_,
                                 a_err_pct=100 * (a_ - a_true) / a_true,
                                 tau_err_pct=100 * (tau_ - tau_true) / tau_true))

df_abl = pd.DataFrame(abl_rows)
os.makedirs("results", exist_ok=True)
df_abl.to_csv("results/switch_ablation.csv", index=False)
print("=== Error in a [%] over five initialisation seeds (same data) ===")
print(df_abl.groupby(['experiment', 'switch'])['a_err_pct'].agg(['mean', 'std', 'min', 'max']).round(2).to_string())

---
## 9 · Systematic Sweep — Error vs Heating Duration

Heating time varied from $0.5\tau$ to $5\tau$ (cooling always $2\tau$). Two random seeds per condition.

In [ ]:
rows2 = []
for heat_tau in [0.5, 1.0, 2.0, 5.0]:
    t_off_ = heat_tau * tau_true
    for seed_ in range(2):
        te_, Te_ = make_experiment(t_off_, t_off_ + 2 * tau_true, seed=100 + seed_)
        for m_name, fn in [
            ('Roper',           roper),
            ('whole-curve fit', whole_curve_fit),
            ('PINN + switch',   lambda t_, T_, to: train_photothermal(t_, T_, to)),
        ]:
            a_, _ = fn(te_, Te_, t_off_)
            rows2.append(dict(heat_tau=heat_tau, method=m_name,
                              a_error=100 * (a_ - a_true) / a_true))

df2 = pd.DataFrame(rows2)
os.makedirs("results", exist_ok=True)
df2.to_csv("results/photothermal_sweep.csv", index=False)

pivot2 = df2.groupby(['heat_tau', 'method'])['a_error'].mean().unstack()
print("=== Mean error in a [%] ===")
print(pivot2.round(1).to_string())

### Figure 2 — Error in $a$ vs $t_{\mathrm{off}}/\tau$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

styles2 = {
    'Roper':           dict(color='tab:orange', marker='s', ls='-',  lw=2),
    'whole-curve fit': dict(color='tab:green',  marker='^', ls='-.', lw=2),
    'PINN + switch':   dict(color='tab:blue',   marker='o', ls=':',  lw=2),
}

for m, sty in styles2.items():
    ax.plot(pivot2.index, pivot2[m], label=m, **sty)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel(r'$t_{\mathrm{off}}\,/\,\tau$')
ax.set_ylabel('Error in $a$ [%]')
ax.set_title('All methods')
ax.legend()

plt.suptitle(r'Relative error in $a$ vs laser heating duration', fontsize=12)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig3_error_sweep.jpg', dpi=150, bbox_inches='tight')
plt.show()

---
## 10 · Validation on Real Experimental Data

### 10.1 Data sources

We searched PubMed Central (PMC) for open-access articles that:
1. Report full heating **and** cooling phases in a single figure
2. Use gold nanoparticles under CW laser irradiation
3. Provide $t_{\mathrm{off}}$ and $T_{\mathrm{env}}$ without ambiguity

All papers are published under **CC BY 4.0** (figure digitisation permitted).

| Dataset | Paper | Figure | PMC ID |
|---|---|---|---|
| GNR 10×38 nm, 808 nm laser | Vikas et al., *Beilstein J. Nanotechnol.* **14**, 205 (2023) | Fig. 6(g) | PMC9924363 |
| GNR 10×41 nm (bare + PEG), 808 nm | Sánchez López de Pablo et al., *Nanoscale Res. Lett.* **9**, 441 (2014) | Fig. 6 | PMC4155392 |
| AuNR and AuNR@Si, 785 nm | Armenta-Gamez et al., *Pharmaceutics* **18**, 310 (2026) | Fig. 4A | PMC13028931 |

See `real_data/SOURCES.md` for full BibTeX citations.

---

### 10.2 Digitisation

Figures were downloaded from the open-access article pages (PMC, MDPI) and digitised with a Python script (Pillow + NumPy): the axes are calibrated from the major tick marks by a linear fit, each curve is extracted as the per-column centroid of its colour, the time origin is moved to the laser switch-on, and the result is linearly interpolated onto a 10 s grid. **No smoothing and no model fit are applied**, so every deviation from the single-$\tau$ ODE in the original figure is kept.

| Error source | Estimate |
|---|---|
| Line centroid / pixel size | ±0.05–0.13 °C, ±2.5–6.6 s (one pixel) |
| Axis calibration (tick-fit residual) | below one pixel (≤ 0.03 °C, ≤ 4.4 s) |
| Switch-on and switch-off times | ±1 pixel column (2.5–6.6 s) |

**Re-check (2026-10-04).** An independent re-digitisation of all five curves from the original figure images (same tick-mark calibration, per-column colour centroid) reproduces the CSV files with an RMS difference of 0.02–0.04 °C (largest single difference 0.06–0.22 °C, near-vertical switch-off segments excluded); the error estimates above are the values measured in that check.

Notes: the tick labelled "35" in Armenta-Gamez Fig. 4A sits at the 30-min position (uniform tick spacing). The bare-GNR baseline is hidden under the other curves, so its switch-on time is taken from the PEG-GNR curve of the same figure. Armenta-Gamez et al. state 25 min of irradiation. On the figure's time axis the laser goes off at 25.4 min (AuNR) and 24.2 min (AuNR@Si), close to that value, but it is switched on at 3.0 and 3.3 min, so the irradiation lasts 22.5 and 20.9 min; this is the $t_{\mathrm{off}}$ used here. A uniform error in the time axis would rescale $a$ and $\tau$ by the same factor and leave $t_{\mathrm{off}}/\tau$, every relative change of $a$ and every residual unchanged (checked with the least-squares fits; the PINN works in $t/t_{\max}$ and is exactly invariant).

---

### 10.3 Experimental conditions

| Dataset | Nanoparticle | Conc. | Laser | $t_{\mathrm{off}}$ (s) | $T_{\mathrm{env}}$ (°C) | $t_{\mathrm{off}}/\tau$ |
|---|---|---|---|---|---|---|
| Vikas 2023 | GNR 10×38 nm | 20 µg/mL | 808 nm CW diode | 900 | 25.0 (ΔT data) | **2.4** |
| Sánchez 2014 — bare GNR | GNR 10×41 nm | 36 µg/mL | 808 nm, 2 W CW | 1799 | 25.47 | **7.9** |
| Sánchez 2014 — PEG-GNR | GNR 10×41 nm (PEG) | 36 µg/mL | 808 nm, 2 W CW | 1799 | 25.34 | **8.2** |
| Armenta-Gamez 2026 — AuNR | AuNR bare | 50 µg/mL | 785 nm, 1.2 W/cm² CW | 1347 | 23.85 | **3.6** |
| Armenta-Gamez 2026 — AuNR@Si | AuNR in mesoporous silica | 50 µg/mL | 785 nm, 1.2 W/cm² CW | 1255 | 24.15 | **5.7** |

**What `ref` means.** The real curves have no known true $a$ or $\tau$: the authors published only the temperature curve. For each curve, `REAL_DATASETS` therefore stores `ref_a` and `ref_tau`, the $a$ and $\tau$ of the **whole-curve least-squares fit** of the single-exponential model (Section 4). They are a yardstick, not the truth: $\tau$ in the last column above is `ref_tau`, and every percentage called "error in $a$" below, including in Figure 3, is the difference from `ref_a`, $100\,(a_{\mathrm{method}} - a_{\mathrm{ref}})/a_{\mathrm{ref}}$. The whole-curve fit is therefore $0.0\,\%$ by definition.

In [ ]:
import os

REAL_DATA_DIR = "real_data"

# t_off and Tenv come from the digitised curves (time origin at laser switch-on);
# ref_a and ref_tau are the whole-curve least-squares fit, used as the reference on real data.
REAL_DATASETS = {
    "Vikas 2023\nGNR 10×38 nm, 808 nm": dict(
        file      = os.path.join(REAL_DATA_DIR, "gnr_10x38_20ugml_NIR_laser.csv"),
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=900.0, Tenv=25.0, ref_a=0.04561, ref_tau=381.0,
    ),
    "Sánchez 2014\nB-GNR 10×41 nm, 808 nm": dict(
        file        = os.path.join(REAL_DATA_DIR, "gnr_10x41_peg_bare_808nm_2W.csv"),
        nanoparticle= "B-GNR_10x41nm_808nm_2W",
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1799.2, Tenv=25.47, ref_a=0.08740, ref_tau=228.4,
    ),
    "Sánchez 2014\nPEG-GNR 10×41 nm, 808 nm": dict(
        file        = os.path.join(REAL_DATA_DIR, "gnr_10x41_peg_bare_808nm_2W.csv"),
        nanoparticle= "PEG-GNR_10x41nm_808nm_2W",
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1799.2, Tenv=25.34, ref_a=0.11486, ref_tau=220.2,
    ),
    "Armenta-Gamez 2026\nAuNR, 785 nm": dict(
        file      = os.path.join(REAL_DATA_DIR, "aunr_785nm_1p2Wcm2_pmc13028931.csv"),
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1347.4, Tenv=23.85, ref_a=0.10533, ref_tau=377.0,
    ),
    "Armenta-Gamez 2026\nAuNR@Si, 785 nm": dict(
        file      = os.path.join(REAL_DATA_DIR, "aunrsi_785nm_1p2Wcm2_pmc13028931.csv"),
        t_col='time_s', T_col='T_C', laser_col='laser_on',
        t_off=1255.1, Tenv=24.15, ref_a=0.09278, ref_tau=220.9,
    ),
}

def load_real(ds):
    df = pd.read_csv(ds["file"])
    if "nanoparticle" in ds:
        df = df[df["nanoparticle"] == ds["nanoparticle"]]
    return df[ds["t_col"]].values.astype(float), df[ds["T_col"]].values.astype(float)

In [ ]:
real_rows, real_fits = [], {}

METHOD_FNS = [
    ("Roper",           lambda t_, T_, to, Tv: roper(t_, T_, to, Tenv_=Tv)),
    ("whole-curve fit", lambda t_, T_, to, Tv: whole_curve_fit(t_, T_, to, Tenv_=Tv)),
    ("PINN + switch",   lambda t_, T_, to, Tv: train_photothermal(t_, T_, to, Tenv_=Tv)),
]

for ds_name, ds in REAL_DATASETS.items():
    t_r, T_r   = load_real(ds)
    t_off_r    = ds["t_off"]
    Tenv_r     = ds["Tenv"]
    ref_a, ref_tau = ds["ref_a"], ds["ref_tau"]

    label = ds_name.replace('\n', ' | ')
    print(f"\n{'─'*60}\n  {label}")
    print(f"  ref: a={ref_a:.4f} °C/s   τ={ref_tau:.0f} s   t_off/τ={t_off_r/ref_tau:.1f}")

    for m_name, fn in METHOD_FNS:
        a_e, tau_e = fn(t_r, T_r, t_off_r, Tenv_r)
        da = 100 * (a_e - ref_a) / ref_a
        dt = 100 * (tau_e - ref_tau) / ref_tau
        rms = np.sqrt(np.mean((T_r - heating_cooling(t_r, a_e, tau_e, t_off_r, Tenv_r)) ** 2))
        print(f"  {m_name:22s}  a={a_e:.4f} ({da:+.1f}%)   τ={tau_e:.0f} ({dt:+.1f}%)   RMS={rms:.2f} °C")
        real_fits[(ds_name, m_name)] = (a_e, tau_e)
        real_rows.append(dict(
            dataset=ds_name.replace('\n', ' '), method=m_name,
            a_est=round(a_e, 5), tau_est=round(tau_e, 1),
            a_err_pct=round(da, 1), tau_err_pct=round(dt, 1), rms_C=round(rms, 2),
            t_off_over_tau=round(t_off_r / ref_tau, 2),
        ))

df_real = pd.DataFrame(real_rows)
os.makedirs("results", exist_ok=True)
df_real.to_csv("results/real_data_results.csv", index=False)

print("\n=== Summary: error in a [%] ===")
print(df_real.pivot(index='method', columns='dataset', values='a_err_pct').to_string())
print("\n=== Residual RMS of the fitted ODE curve [°C] ===")
print(df_real.pivot(index='method', columns='dataset', values='rms_C').to_string())

### Figure 3 — Real data: method comparison

Grey: digitised data. Dashed black: the reference curve (`ref_a`, `ref_tau`). Coloured: the three methods. The percentage in each legend entry is the difference of that method's $a$ from `ref_a`, not a distance from the data. The distance from the data is the RMS in the table printed above (0.36–1.64 °C for the whole-curve fit). The whole-curve fit and the linear-loss PINN use the same single-exponential model, so their curves overlap and their $a$ differ by less than 1 %; both leave the same misfit because a single exponential cannot follow the real curves, which is what Section 11 addresses.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()
axes_flat[-1].set_visible(False)

method_styles = {
    "Roper":           dict(color='tab:orange', ls='--', lw=1.8),
    "whole-curve fit": dict(color='tab:green',  ls='-.', lw=1.8),
    "PINN + switch":   dict(color='tab:blue',   ls=':',  lw=2.2),
}

for ax, (ds_name, ds) in zip(axes_flat, REAL_DATASETS.items()):
    t_r, T_r    = load_real(ds)
    t_off_r     = ds["t_off"]
    Tenv_r      = ds["Tenv"]
    ref_a, ref_tau = ds["ref_a"], ds["ref_tau"]
    tt = np.linspace(0, t_r.max(), 1000)

    ax.plot(t_r / 60, T_r, color='lightgray', lw=1.0, zorder=1, label='data')
    ax.plot(tt / 60, heating_cooling(tt, ref_a, ref_tau, t_off_r, Tenv_r),
            'k--', lw=1.2, alpha=0.6,
            label=f'ref  a={ref_a:.4f}, τ={ref_tau:.0f} s', zorder=2)

    for m_name, sty in method_styles.items():
        a_e, tau_e = real_fits[(ds_name, m_name)]
        da = 100 * (a_e - ref_a) / ref_a
        ax.plot(tt / 60, heating_cooling(tt, a_e, tau_e, t_off_r, Tenv_r),
                label=f'{m_name}  ({da:+.1f}%)', zorder=3, **sty)

    ax.axvline(t_off_r / 60, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('T (°C)')
    ax.set_title(ds_name.replace('\n', '\n'), fontsize=9)
    ax.legend(fontsize=6.5)

plt.suptitle('Real data validation — error in $a$ relative to the whole-curve reference', fontsize=11)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig4_real_data.jpg', dpi=150, bbox_inches='tight')
plt.show()

---
## 11 · Nonlinear Heat Loss: Learning the Loss Law

The single-τ ODE fits the real curves only approximately (Section 10). Their shape — a fast initial rise, a fast initial drop after switch-off, and a slow cooling tail — points to heat losses that grow faster than linearly with $\theta = T - T_{\mathrm{env}}$; laminar natural convection, for example, gives a heat flux $\propto \theta^{5/4}$. We therefore generalise the model to

$$\frac{d\theta}{dt} = a\,L(t) - f(\theta), \qquad f(0) = 0,$$

and compare four estimators of the heating rate $a$ (the quantity the PCE needs):

1. **Roper** — cooling-tail fit (assumes $f(\theta) = \theta/\tau$)
2. **Whole-curve single-τ fit** — Section 4 ($f(\theta) = \theta/\tau$)
3. **Power-law least squares** — $f(\theta) = k\,\theta^{n}$, fitted to the numerical ODE solution (multi-start in $n$). In the code it is parametrised as $f = k_{\mathrm{ref}}\,\theta_{\mathrm{ref}}\,(\theta/\theta_{\mathrm{ref}})^{n}$ with $\theta_{\mathrm{ref}} = \max\theta/2$, i.e. $k = k_{\mathrm{ref}}\,\theta_{\mathrm{ref}}^{\,1-n}$
4. **PINN with a learned loss law** — $f(\theta) = \theta\cdot\mathrm{softplus}(h_\phi(\theta))$, where $h_\phi$ is a small MLP (2 × 20 tanh) trained jointly with the temperature network and $a$; no functional form is assumed

$a$ stays identifiable for any $f$: at the same $\theta$, the heating slope minus the cooling slope equals $a$.

To compare the shape of $f$ across curves we use the **effective loss rate** $r(\theta) = f(\theta)/\theta$ (unit s$^{-1}$, equal to $1/\tau$ for a linear loss). It is not the scaled $\hat\kappa$ of Section 5, which exists only for the linear loss.

**`simulate_loss(t, a_, f, t_off_)`** — numerical solution of $d\theta/dt = a\,L(t) - f(\theta)$, $\theta(0) = 0$ (Section 11)

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | times at which $\theta$ is returned (array) |
| `a_` | $a$ | °C/s | heating rate |
| `f` | $f$ | °C/s | heat-loss law, a function of $\theta$ in °C |
| `t_off_` | $t_{\mathrm{off}}$ | s | laser switch-off time |

**Returns:** the temperature rise $\theta(t)$ in °C.

In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares


def simulate_loss(t, a_, f, t_off_):
    # numerical solution of dθ/dt = a L(t) - f(θ), θ(0) = 0
    rhs = lambda tt, y: [a_ * (tt < t_off_) - f(max(y[0], 0.0))]
    return solve_ivp(rhs, (0, t.max()), [0.0], t_eval=t, max_step=5.0, rtol=1e-8, atol=1e-10).y[0]

**`single_tau_fit(t, th, t_off_)`** — least-squares fit of the linear loss $f = k\theta$, with $k = 1/\tau$, to the numerical solution

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | measured times (array) |
| `th` | $\theta$ | °C | measured temperature rise $T - T_{\mathrm{env}}$ (array) |
| `t_off_` | $t_{\mathrm{off}}$ | s | laser switch-off time |

**Returns:** `([a, k], rms)` — heating rate (°C/s), $k = 1/\tau$ (s$^{-1}$), and the residual RMS (°C).

In [ ]:
def single_tau_fit(t, th, t_off_):
    res = least_squares(lambda p: simulate_loss(t, p[0], lambda x: p[1] * x, t_off_) - th, [0.05, 3e-3],
                        bounds=([1e-5, 1e-6], [5, 1]), diff_step=1e-3, x_scale="jac")
    return res.x, np.sqrt(np.mean(res.fun ** 2))

**`power_law_fit(t, th, t_off_)`** — least-squares fit of $f = k_{\mathrm{ref}}\,\theta_{\mathrm{ref}}\,(\theta/\theta_{\mathrm{ref}})^{n}$ with $\theta_{\mathrm{ref}} = \max\theta/2$, equivalent to $f = k\,\theta^{n}$ with $k = k_{\mathrm{ref}}\,\theta_{\mathrm{ref}}^{\,1-n}$

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | measured times (array) |
| `th` | $\theta$ | °C | measured temperature rise (array) |
| `t_off_` | $t_{\mathrm{off}}$ | s | laser switch-off time |

**Returns:** `([a, k_ref, n], rms)` — heating rate (°C/s), $k_{\mathrm{ref}}$ (s$^{-1}$), exponent $n$, and the residual RMS (°C). Starting exponents: 0.8, 1, 1.3, 1.6, 2 (best fit kept).

In [ ]:
def power_law_fit(t, th, t_off_):
    # f(θ) = k_ref·θref·(θ/θref)^n, i.e. k = k_ref·θref^(1-n); θref decorrelates k_ref and n, several starting exponents avoid local minima
    p_lin, _ = single_tau_fit(t, th, t_off_)
    ref, best = th.max() / 2, None
    for n0 in (0.8, 1.0, 1.3, 1.6, 2.0):
        res = least_squares(lambda p: simulate_loss(t, p[0], lambda x: p[1] * ref * (x / ref) ** p[2], t_off_) - th,
                            [p_lin[0], p_lin[1], n0], bounds=([1e-5, 1e-6, 0.3], [5, 1, 4]), diff_step=1e-3, x_scale="jac")
        if best is None or res.cost < best.cost:
            best = res
    return best.x, np.sqrt(np.mean(best.fun ** 2))

**`LossNet(n)`** — learned loss law $f(\theta) = \theta\cdot\mathrm{softplus}(h_\phi(\theta))$, with $\mathrm{softplus}(x) = \ln(1 + e^{x}) > 0$

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `n` | — | — | neurons per hidden layer of $h_\phi$ (default 20; 2 hidden layers, tanh) |

**Forward:** takes $\hat\theta$ of shape (N, 1) and returns $f$ (scaled) of shape (N, 1); $f = 0$ at $\theta = 0$.

In [ ]:
class LossNet(nn.Module):
    # f(θ) = θ·softplus(h(θ)): zero at ambient temperature, positive above it
    def __init__(self, n=20):
        super().__init__()
        self.h = nn.Sequential(nn.Linear(1, n), nn.Tanh(), nn.Linear(n, n), nn.Tanh(), nn.Linear(n, 1))

    def forward(self, th):
        return th * nn.functional.softplus(self.h(th))

**`LossPINN(t_off_hat, n_units)`** — temperature network with the switch input, plus `loss_law` (a `LossNet`) and the learnable heating rate $\hat a$ (`a`)

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t_off_hat` | $\hat t_{\mathrm{off}}$ | — | scaled switch-off time, $t_{\mathrm{off}}/t_{\mathrm{scale}}$ |
| `n_units` | — | — | neurons per hidden layer of the temperature network (default 100; 4 hidden layers, tanh) |

**Forward:** takes $\hat t$ of shape (N, 1) and returns $\hat\theta$ of shape (N, 1).

In [ ]:
class LossPINN(nn.Module):
    # temperature network with the switch input of Section 5, plus the learned loss law and the heating rate a
    def __init__(self, t_off_hat, n_units=100):
        super().__init__()
        self.t_off_hat = t_off_hat
        layers, d = [], 2
        for _ in range(4):
            layers += [nn.Linear(d, n_units), nn.Tanh()]
            d = n_units
        self.body = nn.Sequential(*layers, nn.Linear(n_units, 1))
        self.loss_law = LossNet()
        self.a = nn.Parameter(torch.tensor([1.0]))

    def forward(self, t):
        return self.body(torch.cat([t, torch.relu(t - self.t_off_hat)], dim=1))

**`train_loss_pinn(t, th, t_off_, adam_epochs, lbfgs_iters, data_weight, physics_weight, seed, n_units, return_terms, log)`** — trains `LossPINN` with data + ODE residual $\hat a\,L - f(\hat\theta) - d\hat\theta/d\hat t$ + initial condition

| Parameter | Symbol | Unit | Meaning |
|---|---|---|---|
| `t` | $t$ | s | measured times (array) |
| `th` | $\theta$ | °C | measured temperature rise $T - T_{\mathrm{env}}$ (array) |
| `t_off_` | $t_{\mathrm{off}}$ | s | laser switch-off time |
| `adam_epochs`, `lbfgs_iters` | — | — | Adam steps (8000) and the ceiling on L-BFGS iterations (1000) |
| `data_weight` | $w_{\mathrm{data}}$ | — | weight of the data loss (default 1) |
| `physics_weight` | $\lambda$ | — | weight of the physics loss, ODE residual + initial condition (default 1) |
| `seed` | — | — | random seed for the weights |
| `n_units` | — | — | width of the four hidden layers of the temperature network (default 100) |
| `return_terms` | — | — | also return the data and physics losses at the end of training (default `False`) |
| `log` | — | — | optional `TrainLog` (Section 12) that records the losses, the gradients and the cost of the run (default `None`) |

**Returns:** `(a_est, f_est, fit)` — heating rate (°C/s), a function `f_est(theta)` giving the learned loss $f$ in °C/s for $\theta$ in °C, and the fitted $\theta$ at the data times (°C).

In [ ]:
def train_loss_pinn(t, th, t_off_, adam_epochs=8000, lbfgs_iters=1000, data_weight=1.0, physics_weight=1.0, seed=0, n_units=100, return_terms=False, log=None):
    torch.manual_seed(seed)
    t_scale, T_scale = float(t.max()), float(th.max())
    t_hat = np_to_th(t / t_scale); theta_hat = np_to_th(th / T_scale)
    model = LossPINN(t_off_ / t_scale, n_units=n_units).to(DEVICE)
    t_col = torch.linspace(0, 1, 800).view(-1, 1).to(DEVICE).requires_grad_(True)
    laser = (t_col < t_off_ / t_scale).float()

    def losses():                      # data loss and physics loss (ODE residual + initial condition)
        th_c = model(t_col)
        dth = grad(th_c, t_col)[0]
        res = dth - model.a * laser + model.loss_law(th_c)
        ic = model(torch.zeros(1, 1).to(DEVICE)).pow(2).mean()
        return ((model(t_hat) - theta_hat) ** 2).mean(), (res ** 2).mean() + ic

    def total_loss(phase):
        data_loss, physics_loss = losses()
        l = data_weight * data_loss + physics_weight * physics_loss
        l.backward()
        if log is not None:
            log.record(phase, data_loss, physics_loss)
        return l

    if log is not None:
        log.start(model, T_scale / t_scale)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(adam_epochs):
        opt.zero_grad(); total_loss("Adam"); opt.step()
    if log is not None:
        log.end_adam()
    opt2 = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iters, line_search_fn="strong_wolfe")
    def closure():
        opt2.zero_grad(); return total_loss("L-BFGS")
    opt2.step(closure)
    if log is not None:
        log.finish(opt2)

    a_est = model.a.item() * T_scale / t_scale
    f_est = lambda x: model.loss_law(np_to_th(np.atleast_1d(x) / T_scale)).detach().cpu().numpy().ravel() * T_scale / t_scale
    fit = model(t_hat).detach().cpu().numpy().ravel() * T_scale
    if return_terms:   # the two losses at the end of training (before the weights)
        data_loss, physics_loss = losses()
        return a_est, f_est, fit, dict(data=data_loss.item(), physics=physics_loss.item())
    return a_est, f_est, fit

**`LOSS_LAWS`, `TRUE_K`, `synthetic_case(law, t_off_, seed_)`** — the two nonlinear loss laws of Section 11.1, their true parameters, and one noisy synthetic curve (times in s, $\theta$ in °C).

**`learned_f_rows(case, t_off_, seed_, th, a_n, f_n, p_s, p_p)`** — table of the learned $f$ on 50 values of $\theta$ from $0.1\,\theta_{\max}$ to $\theta_{\max}$, with the PINN's $a$ and the single-exponential (`p_s`) and power-law (`p_p`) fits of the same curve, for Section 12.5.

In [ ]:
# a_true (heating rate) is the global defined in Section 3
LOSS_LAWS = {
    "power n=1.25":     lambda x: (0.05 / 20 ** 1.25) * np.maximum(x, 0) ** 1.25,
    "linear+quadratic": lambda x: 1.25e-3 * np.maximum(x, 0) + 6.25e-5 * np.maximum(x, 0) ** 2,
}
TRUE_K = {"power n=1.25": {"k_power": 0.05 / 20 ** 1.25, "n": 1.25},
          "linear+quadratic": {"k1": 1.25e-3, "k2": 6.25e-5}}


def synthetic_case(law, t_off_, seed_):
    rng = np.random.default_rng(10 * seed_ + int(t_off_))
    t_ = np.arange(0, 2 * t_off_ + 1, 10.0)
    return t_, simulate_loss(t_, a_true, LOSS_LAWS[law], t_off_) + rng.normal(0, 0.1, t_.size)


def learned_f_rows(case, t_off_, seed_, th, a_n, f_n, p_s=(np.nan, np.nan), p_p=(np.nan, np.nan, np.nan)):
    theta = np.linspace(0.1, 1.0, 50) * th.max()
    return pd.DataFrame({"case": case, "t_off": t_off_, "seed": seed_, "theta": theta, "f": f_n(theta), "a_PINN": a_n,
                         "a_single": p_s[0], "k_single": p_s[1], "a_power": p_p[0],
                         "k_power": p_p[1] * (th.max() / 2) ** (1 - p_p[2]), "n_power": p_p[2]})


### 11.1 Synthetic check with a known loss law

Same $a = 0.05$ °C/s and 0.1 °C noise as before, with two nonlinear loss laws of similar strength: a pure power law $f = k\theta^{1.25}$ (steady state at 20 °C) and a mixed law $f = k_1\theta + k_2\theta^2$ that no single power law represents exactly. Heating lasts 600 s or 1200 s and cooling as long again; three noise seeds per case. The learned $f$ of every run is saved in `results/learned_f_synthetic.csv` for Section 12.5.

In [ ]:
rows3, f_rows = [], []
for law, f_true in LOSS_LAWS.items():
    for t_off_ in (600.0, 1200.0):
        for seed_ in range(3):
            t_, th_ = synthetic_case(law, t_off_, seed_)
            a_r, _ = roper(t_, th_ + Tenv_true, t_off_)
            p_s, rms_s = single_tau_fit(t_, th_, t_off_)
            p_p, rms_p = power_law_fit(t_, th_, t_off_)
            a_n, f_n, fit_n = train_loss_pinn(t_, th_, t_off_, seed=seed_)
            f_rows.append(learned_f_rows(law, t_off_, seed_, th_, a_n, f_n, p_s, p_p))
            grid_ = np.linspace(1, th_.max(), 30)
            rows3.append(dict(law=law, t_off=t_off_, seed=seed_,
                              Roper=100 * (a_r - a_true) / a_true, single_tau=100 * (p_s[0] - a_true) / a_true,
                              power_law=100 * (p_p[0] - a_true) / a_true, loss_PINN=100 * (a_n - a_true) / a_true,
                              n_power=p_p[2], rms_single=rms_s, rms_power=rms_p,
                              rms_PINN=np.sqrt(np.mean((fit_n - th_) ** 2)),
                              f_err_pct=100 * np.max(np.abs(f_n(grid_) - f_true(grid_))) / f_true(th_.max())))

df3 = pd.DataFrame(rows3)
df3.to_csv("results/loss_law_synthetic.csv", index=False)
pd.concat(f_rows).to_csv("results/learned_f_synthetic.csv", index=False)
cols = ["Roper", "single_tau", "power_law", "loss_PINN", "n_power", "rms_single", "rms_power", "rms_PINN", "f_err_pct"]
print("=== Error in a [%], fitted exponent, RMS [°C], max error of the learned f [%] (mean over 3 seeds) ===")
print(df3.groupby(["law", "t_off"])[cols].mean().round(2).to_string())
print("\n=== Standard deviation over seeds ===")
print(df3.groupby(["law", "t_off"])[["Roper", "single_tau", "power_law", "loss_PINN"]].std().round(2).to_string())

### 11.2 Real curves

The same estimators on the five digitised curves. The learned loss law is reported as the effective relaxation rate $r(\theta) = f(\theta)/\theta$ ($= 1/\tau$ for a linear loss); three training seeds show how stable the PINN estimate is. The learned $f$ of every run is saved in `results/learned_f_real.csv` for Section 12.5. Labels: DS1 = Vikas 2023, DS2a/DS2b = Sánchez 2014 bare/PEG-GNR, DS3/DS4 = Armenta-Gamez 2026 AuNR/AuNR@Si.

In [ ]:
DS_LABELS = ["DS1", "DS2a", "DS2b", "DS3", "DS4"]
rows4, loss_curves, f_rows = [], {}, []
for label, (ds_name, ds) in zip(DS_LABELS, REAL_DATASETS.items()):
    t_r, T_r = load_real(ds)
    th_r, t_off_r = T_r - ds["Tenv"], ds["t_off"]
    p_s, rms_s = single_tau_fit(t_r, th_r, t_off_r)
    p_p, rms_p = power_law_fit(t_r, th_r, t_off_r)
    runs = [train_loss_pinn(t_r, th_r, t_off_r, seed=s) for s in range(3)]
    f_rows += [learned_f_rows(label, t_off_r, s, th_r, r[0], r[1], p_s, p_p) for s, r in enumerate(runs)]
    a_runs = np.array([r[0] for r in runs])
    rms_runs = np.array([np.sqrt(np.mean((r[2] - th_r) ** 2)) for r in runs])
    grid_ = np.linspace(0.1, 1.0, 50) * th_r.max()
    loss_curves[ds_name] = (grid_ / th_r.max(), np.array([r[1](grid_) / grid_ for r in runs]))
    rows4.append(dict(dataset=ds_name.replace("\n", " "),
                      a_single=round(p_s[0], 5), a_power=round(p_p[0], 5), n_power=round(p_p[2], 2),
                      a_PINN=round(a_runs.mean(), 5), a_PINN_sd=round(a_runs.std(ddof=1), 5),
                      rms_single=round(rms_s, 2), rms_power=round(rms_p, 2), rms_PINN=round(rms_runs.mean(), 2),
                      r_ratio=round(np.mean([r[1](np.array([0.9 * th_r.max()]))[0] / r[1](np.array([0.2 * th_r.max()]))[0] * 0.2 / 0.9 for r in runs]), 2)))

df4 = pd.DataFrame(rows4)
df4.to_csv("results/loss_law_real.csv", index=False)
pd.concat(f_rows).to_csv("results/learned_f_real.csv", index=False)
print("a in °C/s; RMS in °C; r_ratio = r(0.9 θmax) / r(0.2 θmax) of the learned law (1 for a linear loss)")
print(df4.to_string(index=False))

# learned r(θ) normalised at 0.2 θmax, per curve (mean and range over the three seeds), saved for re-plotting
curve_rows = []
for label, (x, r_runs) in zip(DS_LABELS, loss_curves.values()):
    r_norm = r_runs / r_runs[:, [np.argmin(np.abs(x - 0.2))]]
    curve_rows.append(pd.DataFrame({"dataset": label, "theta_over_max": x, "r_mean": r_norm.mean(axis=0),
                                    "r_min": r_norm.min(axis=0), "r_max": r_norm.max(axis=0)}))
df_curves = pd.concat(curve_rows)
df_curves.to_csv("results/loss_law_curves.csv", index=False)

# fixed categorical order (validated reference palette) plus distinct line styles for black-and-white print
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
STYLES = ["-", "--", "-.", ":", (0, (5, 1, 1, 1, 1, 1))]
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(3.4, 2.3))
    for (label, d), color, ls in zip(df_curves.groupby("dataset", sort=False), COLORS, STYLES):
        ax.fill_between(d.theta_over_max, d.r_min, d.r_max, color=color, alpha=0.15, lw=0)
        ax.plot(d.theta_over_max, d.r_mean, color=color, ls=ls, lw=1.6, label=label)
    ax.axhline(1, color="0.45", lw=0.8)
    ax.set_xlabel(r"$\theta / \theta_{\max}$", fontsize=9)
    ax.set_ylabel(r"$r(\theta)\,/\,r(0.2\,\theta_{\max})$", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.3, lw=0.5)
    ax.legend(fontsize=7.5, frameon=False, ncol=2, loc="upper left")
    fig.tight_layout()
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/fig5_learned_loss_law.jpg", dpi=200, bbox_inches="tight")
    fig.savefig("results/plots/fig5_learned_loss_law.pdf", bbox_inches="tight")
    plt.show()


### 11.3 Observations

1. **Synthetic data.** With a nonlinear loss, Roper misses $a$ by −29 % to −34 % even for 1200 s of heating (−33 % to −37 % for 600 s), and the single-exponential fit by up to −2.5 %. The power-law fit and the learned-loss PINN both recover $a$ within 0.4 %; the learned law matches the true $f$ within 2.9 %, while the power law can describe the mixed law only through an effective exponent of 1.36.
2. **Real data — residual.** The learned loss law gives the lowest residual on all five curves: 0.23–0.68 °C, against 0.36–1.64 °C for the single exponential and 0.24–0.83 °C for the power law (19–82 % below the single-exponential misfit).
3. **Real data — shape of the loss.** The effective loss rate $r(\theta) = f(\theta)/\theta$ grows with temperature on every curve ($r(0.9\,\theta_{\max}) / r(0.2\,\theta_{\max})$ = 1.34–2.90). Three curves give power-law exponents of 1.18–1.31, close to laminar natural convection ($n = 5/4$); DS2b gives 1.62 and DS3 1.01.
4. **Real data — heating rate.** The learned-loss $a$ differs from the single-exponential one by −0.7 % to +11.6 %; the three largest changes (+5.8 %, +4.1 %, +11.6 %) are on the three most nonlinear curves (DS2a, DS4, DS2b). The spread over three seeds is at most 0.6 % (DS2a). DS4 is not well determined (point 6).
5. **Caveat.** The learned $f$ is an *effective* loss law: besides convection and radiation it can absorb sensor effects, such as the fast transients at switch-on and switch-off in Armenta-Gamez Fig. 4A.
6. **DS4 is not well determined (Section 11.5).** Over 15 runs (widths 20, 50, 100; five seeds) the change of $a$ relative to the single exponential spans +2.1 % to +14.0 %: 11 runs give < 5 %, one +5.0 %, one +8.3 %, two +13 % to +14 %. The correlation between the ODE misfit and the change is −0.98, and the run with the lowest misfit (0.53 °C) gives +13.2 %, so the reported mean of three seeds (+4.1 %) is not reliable.
7. **Held-out extrapolation (Section 11.4).** The learned law predicts the held-out cooling tail better than the power-law fit on DS2a, DS2b and DS4 (0.23, 0.12, 0.75 °C against 0.73, 0.82, 1.60 °C; seed SD 0.13, 0.03, 0.71 °C) and worse on DS1 and DS3 (0.92 and 1.96 °C against 0.19 and 0.61 °C). Its advantage therefore holds only for strongly nonlinear curves; why it fails on the others was not tested.

### 11.4 Held-out extrapolation

A lower residual on the fitted data does not prove that a loss law is correct, since a more flexible model also fits noise better. For each curve, every method is therefore trained on the heating phase and the first half of the cooling phase only, and the second half of the cooling phase is predicted by integrating the ODE with the fitted $a$ and loss law (the PINN's collocation points also stay inside the training window). The table gives the RMS error on the held-out points in °C (the PINN over three seeds), saved in `results/loss_law_extrapolation.csv`.

**Result.** The learned law gives a smaller held-out error than the power-law fit on DS2a, DS2b and DS4 (0.23, 0.12, 0.75 °C against 0.73, 0.82, 1.60 °C) and a larger one on DS1 and DS3 (0.92 and 1.96 °C against 0.19 and 0.61 °C); the single exponential gives 0.55, 1.25, 2.35, 0.22 and 1.81 °C on DS1, DS2a, DS2b, DS3, DS4.

In [ ]:
# Section 11.4: train on heating + first half of cooling, predict the second half of cooling
rows5 = []
for ds_name, ds in REAL_DATASETS.items():
    t_r, T_r = load_real(ds)
    th_r, t_off_r = T_r - ds["Tenv"], ds["t_off"]
    cut = t_off_r + 0.5 * (t_r.max() - t_off_r)
    fit, held = t_r <= cut, t_r > cut
    rms_held = lambda y: np.sqrt(np.mean((y[held] - th_r[held]) ** 2))

    p_s, _ = single_tau_fit(t_r[fit], th_r[fit], t_off_r)
    p_p, _ = power_law_fit(t_r[fit], th_r[fit], t_off_r)
    ref = th_r[fit].max() / 2                                   # theta_ref used inside power_law_fit
    y_s = simulate_loss(t_r, p_s[0], lambda x: p_s[1] * x, t_off_r)
    y_p = simulate_loss(t_r, p_p[0], lambda x: p_p[1] * ref * (x / ref) ** p_p[2], t_off_r)

    rms_pinn = []
    for seed_ in range(3):
        a_n, f_n, _ = train_loss_pinn(t_r[fit], th_r[fit], t_off_r, seed=seed_)
        y_n = simulate_loss(t_r, a_n, lambda x: float(f_n(np.array([x]))[0]), t_off_r)
        rms_pinn.append(rms_held(y_n))

    rows5.append(dict(dataset=ds_name.replace("\n", " "), n_fit=int(fit.sum()), n_held=int(held.sum()),
                      rms_single=rms_held(y_s), rms_power=rms_held(y_p),
                      rms_PINN=np.mean(rms_pinn), rms_PINN_sd=np.std(rms_pinn, ddof=1)))

df5 = pd.DataFrame(rows5)
df5.to_csv("results/loss_law_extrapolation.csv", index=False)
print("RMS error on the held-out cooling tail [°C]")
print(df5.round(2).to_string(index=False))

### 11.5 Is the heating rate of DS4 well determined?

On DS4 the learned loss law can end in different solutions. Here the same curve is trained with three network widths and five seeds each. `da_vs_single_pct` is the change of the learned $a$ relative to the single-exponential fit, `rms_ode` the RMS misfit of the ODE solution with the learned $a$ and $f$, and `rms_network` the RMS misfit of the temperature network itself. The table is saved in `results/loss_law_ds4_sensitivity.csv`.

**Result.** `da_vs_single_pct` spans +2.1 % to +14.0 %, and it falls as the misfit falls (correlation of `rms_ode` with `da_vs_single_pct`: −0.98); the lowest misfit (0.53 °C) gives +13.2 %.

In [ ]:
# Section 11.5: DS4 with several network widths and seeds
ds4_name, ds4 = list(REAL_DATASETS.items())[-1]
t_r, T_r = load_real(ds4)
th_r, t_off_r = T_r - ds4["Tenv"], ds4["t_off"]
p_s, _ = single_tau_fit(t_r, th_r, t_off_r)

rows6 = []
for n_units_ in (20, 50, 100):
    for seed_ in range(5):
        a_n, f_n, fit_n = train_loss_pinn(t_r, th_r, t_off_r, seed=seed_, n_units=n_units_)
        y_n = simulate_loss(t_r, a_n, lambda x: float(f_n(np.array([x]))[0]), t_off_r)
        rows6.append(dict(n_units=n_units_, seed=seed_, a=a_n,
                          da_vs_single_pct=100 * (a_n - p_s[0]) / p_s[0],
                          rms_network=np.sqrt(np.mean((fit_n - th_r) ** 2)),
                          rms_ode=np.sqrt(np.mean((y_n - th_r) ** 2))))

df6 = pd.DataFrame(rows6)
df6.to_csv("results/loss_law_ds4_sensitivity.csv", index=False)
print(ds4_name.replace("\n", " "), "| single-exponential a =", round(p_s[0], 5), "°C/s")
print(df6.round(4).to_string(index=False))

### 11.6 Fitted curves on the real data

In Figure 3 (Section 10) every method is drawn as the closed-form single-exponential curve with its estimated $a$ and $\tau$, so the whole-curve fit and the linear-loss PINN overlap by construction. Here the PINN curves are the networks' **own outputs** at the data times:

- **linear-loss PINN (network):** the temperature network of Section 5; its physics term ties it to the same single-exponential equation as the whole-curve fit, so it can only follow the data as closely as that equation allows;
- **learned-loss PINN (network):** the temperature network of Section 11, whose loss law is learned, so it can follow the curved shape of the data.

The RMS in each legend is the distance from the data. The networks are retrained here (seed 0), so their $a$ can differ slightly from Sections 10 and 11.2. Curves are saved in `results/real_fits_curves.csv`, the figure in `results/plots/fig6_real_fits.jpg`.

In [ ]:
# Section 11.6: whole-curve fit, linear-loss PINN network output and learned-loss PINN network output on the real curves
from contextlib import redirect_stdout
import io

labels = ["DS1", "DS2a", "DS2b", "DS3", "DS4"]
fit_rows, summary = [], []
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
axes = axes.ravel()
axes[-1].set_visible(False)

for ax, label, (ds_name, ds) in zip(axes, labels, REAL_DATASETS.items()):
    t_r, T_r = load_real(ds)
    t_off_r, Tenv_r = ds["t_off"], ds["Tenv"]
    rms = lambda y: np.sqrt(np.mean((y - T_r) ** 2))

    a_c, tau_c = whole_curve_fit(t_r, T_r, t_off_r, Tenv_=Tenv_r)
    T_cf = heating_cooling(t_r, a_c, tau_c, t_off_r, Tenv_r)
    with redirect_stdout(io.StringIO()):                                # hide the per-epoch log
        a_l, tau_l, T_lin = train_photothermal(t_r, T_r, t_off_r, Tenv_=Tenv_r, return_fit=True)
    a_n, f_n, th_n = train_loss_pinn(t_r, T_r - Tenv_r, t_off_r, seed=0)
    T_learn = th_n + Tenv_r

    fit_rows.append(pd.DataFrame({"dataset": label, "time_s": t_r, "T_data": T_r, "T_whole_curve": T_cf,
                                  "T_linear_PINN_network": T_lin, "T_learned_PINN_network": T_learn}))
    summary.append(dict(dataset=label, a_whole_curve=round(a_c, 5), a_linear_PINN=round(a_l, 5), a_learned_PINN=round(a_n, 5),
                        rms_whole_curve=round(rms(T_cf), 3), rms_linear_PINN_network=round(rms(T_lin), 3),
                        rms_learned_PINN_network=round(rms(T_learn), 3)))

    ax.plot(t_r / 60, T_r, color='0.7', lw=3, label='data')
    ax.plot(t_r / 60, T_cf, color='tab:green', lw=1.6, label=f'whole-curve fit (RMS {rms(T_cf):.2f} °C)')
    ax.plot(t_r / 60, T_lin, color='tab:blue', ls=':', lw=2.2, label=f'linear-loss PINN, network (RMS {rms(T_lin):.2f} °C)')
    ax.plot(t_r / 60, T_learn, color='tab:red', ls='--', lw=1.6, label=f'learned-loss PINN, network (RMS {rms(T_learn):.2f} °C)')
    ax.axvline(t_off_r / 60, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set_title(f"{label}: " + ds_name.replace("\n", " | "), fontsize=9)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('T (°C)')
    ax.legend(fontsize=7)

df_fits = pd.concat(fit_rows)
df_fits.to_csv("results/real_fits_curves.csv", index=False)
print(pd.DataFrame(summary).to_string(index=False))

plt.suptitle('Real data: fitted curves (PINN curves are the networks\' own outputs)', fontsize=11)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/fig6_real_fits.jpg', dpi=150, bbox_inches='tight')
plt.show()

### 11.7 Sensitivity to the weight of the physics loss

The training loss is `data_weight × data loss + physics_weight × physics loss` (the physics loss is the ODE residual plus the initial condition), with `data_weight = 1` and `physics_weight = 1` ($\lambda$ in the paper) in all runs above. Equal weights are the usual starting point, but the PINN literature (for example Wang, Teng and Perdikaris, arXiv:2001.04536) warns that one term can dominate the other and that no weight is universally correct, so the weights were not tuned and its effect has to be shown. Both PINNs are trained here with `physics_weight` = 0.1, 1 and 10 on all five real curves, three seeds each; `physics_weight` = 1 is the pipeline of the earlier sections, repeated here so that all three values sit in one table with the same seeds. `da_vs_single_pct` is the change of the estimated $a$ relative to the single-exponential fit and `rms_network` the RMS misfit of the network's own output. For the learned-loss PINN the table also gives `rms_ode` (the misfit of the ODE solution with the learned $a$ and $f$) and `data_loss` and `physics_loss`, the two losses at the end of training before the weights; they show whether one part is much larger than the other. The table is saved in `results/loss_law_lambda_sensitivity.csv`.

**Result** (means over three seeds; `physics_weight` = 0.1 / 1 / 10).

| Curve | Model | change of $a$ vs single exponential | RMS of the network (°C) |
|---|---|---|---|
| DS1 | learned-loss | −0.5 % / +1.0 % / +1.2 % | 0.18 / 0.23 / 0.24 |
| DS2a | learned-loss | +6.7 % / +5.8 % / +2.4 % | 0.22 / 0.29 / 0.46 |
| DS2b | learned-loss | +12.2 % / +11.6 % / −0.3 % | 0.24 / 0.29 / 0.63 |
| DS3 | learned-loss | +0.3 % / −0.7 % / −0.2 % | 0.33 / 0.46 / 0.53 |
| DS4 | learned-loss | +6.8 % / +4.1 % / +1.8 % | 0.56 / 0.68 / 0.79 |
| DS1 | linear-loss | −1.8 % / −0.3 % / −0.1 % | 0.25 / 0.34 / 0.36 |
| DS2a | linear-loss | −1.2 % / +0.4 % / −16.2 % | 0.73 / 0.78 / 1.20 |
| DS2b | linear-loss | −3.0 % / +0.1 % / −20.2 % | 1.52 / 1.62 / 1.93 |
| DS3 | linear-loss | −0.7 % / −0.1 % / +0.0 % | 0.48 / 0.55 / 0.57 |
| DS4 | linear-loss | −2.5 % / −0.5 % / −0.4 % | 0.83 / 0.95 / 0.97 |

1. **Weights 0.1 and 1 give the same picture for the learned-loss PINN.** The change of $a$ differs between them by at most 1.5 percentage points on DS1, DS2a, DS2b and DS3 and by 2.7 on DS4, and the three nonlinear curves keep the largest changes (DS2b +12.2 % and +11.6 %, DS2a +6.7 % and +5.8 %, DS4 +6.8 % and +4.1 %). The weight-1 values equal the main results.
2. **Weight 10 breaks the learned-loss result on the nonlinear curves.** The network follows the data worse (RMS 0.46, 0.63 and 0.79 °C on DS2a, DS2b and DS4 against 0.29, 0.29 and 0.68 °C at weight 1) and the changes shrink to +2.4 %, −0.3 % and +1.8 %. DS1 does not change (0.24 against 0.23 °C).
3. **A smaller weight fits the data better on every curve** (RMS at weight 0.1: 0.18, 0.22, 0.24, 0.33, 0.56 °C), but on DS4 the seeds then disagree (one gives +14.7 %, two give +2.8 %), so DS4 stays not well determined (Section 11.5).
4. **The linear-loss PINN equals the single exponential only near weight 1** (changes of $a$ of −0.3 % to +0.4 % on all curves). With weight 0.1 it moves by up to −3.0 % (DS2b) and with weight 10 by −16 % on DS2a and −20 % on DS2b, so its agreement with the whole-curve fit (Section 7) depends on this choice.
5. **Balance of the two losses at the end of training** (before the weights; learned-loss, mean): at weight 1 the physics loss is 0.00001–0.00003 against data losses of 0.00014–0.00106, so the data loss is the larger term on all five curves; at weight 0.1 the two are of similar size on DS2a, DS2b and DS4 (for example 0.00010 and 0.00008 on DS2b).
6. Weight 1 is therefore not the best weight for the data fit, but it is the weight at which the seeds agree best and at which the linear-loss PINN reproduces the single exponential, and the nonlinear-curve result holds for 0.1 to 1; the paper keeps weight 1.


In [ ]:
# Section 11.7: sensitivity to the weight of the physics loss (physics_weight) (both PINNs)
from contextlib import redirect_stdout
import io

rows7 = []
for label, (ds_name, ds) in zip(["DS1", "DS2a", "DS2b", "DS3", "DS4"], REAL_DATASETS.items()):   # all five curves
    t_r, T_r = load_real(ds)
    Tenv_r, t_off_r = ds["Tenv"], ds["t_off"]
    th_r = T_r - Tenv_r
    p_s, _ = single_tau_fit(t_r, th_r, t_off_r)
    for w_ in (0.1, 1.0, 10.0):
        for seed_ in range(3):
            with redirect_stdout(io.StringIO()):                                # hide the per-epoch log
                a_l, tau_l, T_lin = train_photothermal(t_r, T_r, t_off_r, Tenv_=Tenv_r, data_weight=1.0, physics_weight=w_,
                                                       seed=seed_, return_fit=True)
            rows7.append(dict(dataset=label, model="linear-loss", physics_weight=w_, seed=seed_, a=a_l,
                              da_vs_single_pct=100 * (a_l - p_s[0]) / p_s[0],
                              rms_network=np.sqrt(np.mean((T_lin - T_r) ** 2))))

            a_n, f_n, fit_n, tm = train_loss_pinn(t_r, th_r, t_off_r, data_weight=1.0, physics_weight=w_, seed=seed_, return_terms=True)
            y_n = simulate_loss(t_r, a_n, lambda x: float(f_n(np.array([x]))[0]), t_off_r)
            rows7.append(dict(dataset=label, model="learned-loss", physics_weight=w_, seed=seed_, a=a_n,
                              da_vs_single_pct=100 * (a_n - p_s[0]) / p_s[0],
                              rms_network=np.sqrt(np.mean((fit_n - th_r) ** 2)),
                              rms_ode=np.sqrt(np.mean((y_n - th_r) ** 2)),
                              data_loss=tm["data"], physics_loss=tm["physics"]))

df7 = pd.DataFrame(rows7)
df7.to_csv("results/loss_law_lambda_sensitivity.csv", index=False)
print(df7.round(4).to_string(index=False))
print()
print(df7.groupby(["dataset", "model", "physics_weight"])[["da_vs_single_pct", "rms_network"]].agg(["min", "max"]).round(3).to_string())


---
## 12 · Training Diagnostics and Parameter Recovery

| Section | Question |
|---|---|
| 12.1 | Where are the collocation points? |
| 12.2 | How do the losses and the gradient of each layer behave during training, and what does each PINN cost? |
| 12.3 | What happens if the learned-loss PINN gets the training budget of the linear-loss PINN? |
| 12.4 | Which curve-fitting methods does the notebook use? |
| 12.5 | Can $k$ be recovered from the learned loss law? |
| 12.6 | How good are the curves rebuilt from the recovered parameters? |

Section 12.5 reads the learned $f$ saved by Sections 11.1 and 11.2, so those two sections must have run with the current code. Every training cell saves a CSV and every plot reads one, so a plot can be redrawn without training.

**`TrainLog(every)`** — optional recorder passed as `log=` to `train_photothermal` and `train_loss_pinn`. It records the data loss, the physics loss, the estimated $a$ and the gradient of each layer, every `every` Adam steps and at every L-BFGS evaluation, and the cost of the run. With `every=0` it keeps only the cost.

In [ ]:
import io
import time
from contextlib import redirect_stdout

DS_LABELS = ["DS1", "DS2a", "DS2b", "DS3", "DS4"]
REAL_BY_LABEL = dict(zip(DS_LABELS, REAL_DATASETS.values()))
DIAG_CURVE = "DS2b"      # the curve used in Sections 12.1 and 12.2


def linear_layers(model):
    # the Linear layers in forward order: temperature network T1...T4, Tout, then loss network f1, f2, fout
    named = [(name, m) for name, m in model.named_modules() if isinstance(m, nn.Linear)]
    temp = [m for name, m in named if not name.startswith("loss_law")]
    loss = [m for name, m in named if name.startswith("loss_law")]
    names = [f"T{i + 1}" for i in range(len(temp) - 1)] + ["Tout"]
    if loss:
        names += [f"f{i + 1}" for i in range(len(loss) - 1)] + ["fout"]
    return list(zip(names, temp + loss))


class TrainLog:
    def __init__(self, every=10):
        self.every, self.rows, self.evals = every, [], {"Adam": 0, "L-BFGS": 0}
        self.logging_s, self.seconds = 0.0, {}

    def start(self, model, a_scale):
        self.model, self.a_scale, self.layers = model, a_scale, linear_layers(model)
        self.t0 = time.perf_counter()

    def record(self, phase, data_loss, physics_loss):
        self.evals[phase] += 1
        if self.every == 0 or (phase == "Adam" and (self.evals[phase] - 1) % self.every != 0):
            return
        t_log = time.perf_counter()
        row = dict(phase=phase, n_eval=sum(self.evals.values()), data_loss=data_loss.item(),
                   physics_loss=physics_loss.item(), a=self.model.a.item() * self.a_scale)
        for name, m in self.layers:
            row[name] = torch.cat([m.weight.grad.ravel(), m.bias.grad.ravel()]).pow(2).mean().sqrt().item()
        self.rows.append(row)
        self.logging_s += time.perf_counter() - t_log

    def end_adam(self):
        self.seconds["Adam"] = time.perf_counter() - self.t0 - self.logging_s

    def finish(self, lbfgs):
        self.seconds["L-BFGS"] = time.perf_counter() - self.t0 - self.logging_s - self.seconds["Adam"]
        self.lbfgs_iters = lbfgs.state_dict()["state"][0]["n_iter"]
        self.lbfgs_limits = lbfgs.param_groups[0]["max_iter"], lbfgs.param_groups[0]["max_eval"]

    def cost(self):
        max_iter, max_eval = self.lbfgs_limits
        stop = ("iteration limit" if self.lbfgs_iters >= max_iter
                else "evaluation limit" if self.evals["L-BFGS"] >= max_eval else "tolerance")
        loss_net = self.model.loss_law.parameters() if hasattr(self.model, "loss_law") else []
        return dict(n_params=sum(p.numel() for p in self.model.parameters()),
                    n_params_loss_net=sum(p.numel() for p in loss_net),
                    adam_steps=self.evals["Adam"], lbfgs_iter_limit=max_iter, lbfgs_eval_limit=max_eval,
                    lbfgs_iters=self.lbfgs_iters, lbfgs_evals=self.evals["L-BFGS"], lbfgs_stop=stop,
                    seconds_adam=self.seconds["Adam"], seconds_lbfgs=self.seconds["L-BFGS"])


### 12.1 Collocation points

The ODE residual is evaluated at 800 fixed, evenly spaced times in the scaled range $\hat t \in [0, 1]$, the same way in both PINNs (`t_col` in `train_photothermal` and `train_loss_pinn`), which covers the record from $t = 0$ to the last data time. The points are not resampled during training, and no extra points are placed near $t_{\mathrm{off}}$. The table gives the spacing on each real curve and the number of points in the heating and cooling phases; the figure shows the data and the collocation times on `DIAG_CURVE`, with a zoom around the switch-off. It is saved in `results/plots/fig7_collocation_points.jpg`.

**Result.** On the real curves the collocation spacing is 2.24–4.16 s against 10 s for the data, so the 800 collocation points sit on 179–332 data points. The heating phase holds 402–570 of them and the cooling phase 230–398.

In [ ]:
# Section 12.1: the data times and the 800 collocation times (no training)
rows_c = []
for label, ds in REAL_BY_LABEL.items():
    t_r, _ = load_real(ds)
    t_c = np.linspace(0, 1, 800) * t_r.max()
    rows_c.append(dict(curve=label, n_data=t_r.size, data_spacing_s=np.median(np.diff(t_r)), n_collocation=t_c.size,
                       collocation_spacing_s=t_c[1], n_heating=int((t_c < ds["t_off"]).sum()),
                       n_cooling=int((t_c >= ds["t_off"]).sum())))
print(pd.DataFrame(rows_c).round(2).to_string(index=False))

ds = REAL_BY_LABEL[DIAG_CURVE]
t_r, T_r = load_real(ds)
t_c = np.linspace(0, 1, 800) * t_r.max()
heat = t_c < ds["t_off"]
strip = T_r.min() - 0.08 * np.ptp(T_r)
with plt.style.context("default"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw=dict(width_ratios=[3, 2]))
    for ax in axes:
        ax.plot(t_r / 60, T_r, "o", ms=3, color="0.35", label="data")
        ax.plot(t_c[heat] / 60, np.full(heat.sum(), strip), "|", ms=8, color="#eb6834", label="collocation, laser on")
        ax.plot(t_c[~heat] / 60, np.full((~heat).sum(), strip), "|", ms=8, color="#2a78d6", label="collocation, laser off")
        ax.axvline(ds["t_off"] / 60, color="0.45", lw=0.8)
        ax.set_xlabel("Time (min)")
        ax.set_ylabel("T (°C)")
        ax.grid(alpha=0.3, lw=0.5)
    axes[0].set_title(f"{DIAG_CURVE}: whole record", fontsize=10)
    axes[0].legend(fontsize=8, frameon=False, loc="center right")
    axes[1].set_xlim((ds["t_off"] - 90) / 60, (ds["t_off"] + 90) / 60)
    axes[1].set_title("Zoom around the laser switch-off", fontsize=10)
    fig.tight_layout()
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/fig7_collocation_points.jpg", dpi=150, bbox_inches="tight")
    plt.show()


### 12.2 Training losses, gradient flow and budget

Each PINN is trained on `DIAG_CURVE` (seed 0) with its usual settings and a `TrainLog`. The gradient of a layer is reported as $\|g\|/\sqrt{n}$, the root mean square over its $n$ weights and biases, so that layers of different size can be compared. `T1`–`T4` are the hidden layers of the temperature network counted from the input, `Tout` is its output layer, and `f1`, `f2`, `fout` are the layers of the loss network. A vanishing gradient would appear as a steady fall from layer to layer towards the input (`T4`, `T3`, `T2`, `T1`).

The budget table gives the parameters of each PINN and what its training really used. `lbfgs_iters` and `lbfgs_evals` are counted, because the iteration number passed to L-BFGS is only a ceiling: PyTorch also stops after 5/4 of it in function evaluations and when its tolerances are reached (`lbfgs_stop` says which rule ended the run). The times exclude the logging itself. The results are saved in `results/training_history.csv` and `results/training_budget.csv`, the figures in `results/plots/fig8_training_losses.jpg` and `results/plots/fig9_gradient_flow.jpg`.

**Result** (DS2b, seed 0).

| | Linear-loss PINN | Learned-loss PINN |
|---|---|---|
| Parameters (of which loss network) | 30,703 (0) | 31,183 (481) |
| Adam steps | 3000 | 8000 |
| L-BFGS iterations used / ceiling | 300 / 300 | 811 / 1000 |
| L-BFGS evaluations used / ceiling | 345 / 375 | 939 / 1250 |
| What ended L-BFGS | iteration limit | tolerance |
| Gradient evaluations in total | 3345 | 8939 |
| Time, Adam + L-BFGS (s) | 8.05 + 2.02 | 21.74 + 6.01 |
| Time per evaluation (ms) | 3.01 | 3.10 |
| Total loss at the start, end of Adam, end of L-BFGS | 0.406, 0.0989, 0.00453 | 0.879, 0.0266, 0.000154 |
| Estimated $a$ at the end of Adam, final (°C/s) | 0.0255, 0.1149 | 0.0518, 0.1279 |

1. **Budget.** The learned-loss PINN has 1.6 % more parameters and nearly the same time per evaluation, so its 2.8 times longer run comes from the larger budget, not from the larger model.
2. **Losses.** On this curve Adam ends far from the final $a$, and L-BFGS does most of the work: it lowers the total loss 22-fold in the linear-loss PINN and 172-fold in the learned-loss PINN, and moves $a$ to its final value. In the linear-loss PINN Adam flattens out after about 2000 evaluations: the loss falls only from 0.114 to 0.099, with gradients of 7e-07 to 2e-06 in `T2`–`T4`. The loss also has isolated spikes during Adam, the largest recorded being 1.59 at evaluation 1701 (linear-loss) and 3.53 at evaluation 2571 (learned-loss). The L-BFGS records include the trial points of its line search.
3. **Gradient flow.** There is no vanishing gradient in this run. The gradient of `T1` is 0.61–8.28 times that of `T4`, so it does not fall steadily towards the input. The printed `T1_over_Tout` is below 1 at every moment (0.04–0.29) because the output layer has a larger gradient than all hidden layers (`Tout` is 8.3–116 times `T4`). This is a single step at the last layer, not a decay across the depth. In the loss network, `f1` is 0.58–4.51 times `f2`. These numbers come from one curve and one seed.

In [ ]:
# Section 12.2: one logged training run of each PINN on the same curve
ds = REAL_BY_LABEL[DIAG_CURVE]
t_r, T_r = load_real(ds)
Tenv_r, t_off_r = ds["Tenv"], ds["t_off"]
log_lin, log_learn = TrainLog(), TrainLog()
with redirect_stdout(io.StringIO()):                                # hide the per-epoch log
    train_photothermal(t_r, T_r, t_off_r, Tenv_=Tenv_r, seed=0, log=log_lin)
train_loss_pinn(t_r, T_r - Tenv_r, t_off_r, seed=0, log=log_learn)

hist = pd.concat([pd.DataFrame(log_lin.rows).assign(model="linear-loss"),
                  pd.DataFrame(log_learn.rows).assign(model="learned-loss")], ignore_index=True)
hist.to_csv("results/training_history.csv", index=False)

budget = pd.DataFrame([dict(model="linear-loss", **log_lin.cost()), dict(model="learned-loss", **log_learn.cost())])
budget["ms_per_evaluation"] = 1000 * (budget.seconds_adam + budget.seconds_lbfgs) / (budget.adam_steps + budget.lbfgs_evals)
budget.to_csv("results/training_budget.csv", index=False)
print(budget.round(2).set_index("model").T.to_string())


In [ ]:
# Training losses and estimated a, read from results/training_history.csv
hist = pd.read_csv("results/training_history.csv")
with plt.style.context("default"):
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.2), sharex="col", gridspec_kw=dict(height_ratios=[2, 1]))
    for col, model_name in enumerate(["linear-loss", "learned-loss"]):
        h = hist[hist.model == model_name]
        n_adam = h.loc[h.phase == "L-BFGS", "n_eval"].min() - 1
        top, bottom = axes[0, col], axes[1, col]
        top.plot(h.n_eval, h.data_loss + h.physics_loss, color="#2a78d6", lw=1.4, label="total (both weights 1)")
        top.plot(h.n_eval, h.data_loss, color="#eb6834", ls="--", lw=1.2, label="data")
        top.plot(h.n_eval, h.physics_loss, color="#1baf7a", ls="-.", lw=1.2, label="physics")
        top.set_yscale("log")
        top.set_ylim(top=top.get_ylim()[1] * 5)
        top.set_ylabel("loss")
        top.set_title(f"{model_name} PINN ({DIAG_CURVE}, seed 0)", fontsize=10)
        top.legend(fontsize=8, frameon=False, loc="upper right")
        top.text(n_adam, 0.03, " L-BFGS", transform=top.get_xaxis_transform(), fontsize=8, color="0.3")
        bottom.plot(h.n_eval, h.a, color="#2a78d6", lw=1.4)
        bottom.set_ylim(0, 1.3 * h.a.iloc[-1])
        bottom.set_ylabel("estimated a (°C/s)")
        bottom.set_xlabel("gradient evaluations")
        for ax in (top, bottom):
            ax.axvline(n_adam, color="0.45", lw=0.8)
            ax.grid(alpha=0.3, lw=0.5)
            if model_name == "learned-loss":        # the Adam budget of the linear-loss PINN
                ax.axvline(3000, color="0.45", lw=0.8, ls=":")
    axes[0, 1].text(3000, 0.03, " 3000 steps", transform=axes[0, 1].get_xaxis_transform(), fontsize=8, color="0.3")
    fig.tight_layout()
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/fig8_training_losses.jpg", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# Gradient flow per layer (vanishing-gradient check), read from results/training_history.csv
hist = pd.read_csv("results/training_history.csv")
LAYER_STYLE = {"T1": ("#86b6ef", "-"), "T2": ("#5598e7", "-"), "T3": ("#2a78d6", "-"), "T4": ("#1c5cab", "-"), "Tout": ("#0d366b", "-"),
               "f1": ("#ef9670", "--"), "f2": ("#eb6834", "--"), "fout": ("#a8431c", "--")}
MOMENT_STYLE = {"start of Adam": ("#2a78d6", "o"), "end of Adam": ("#eb6834", "s"), "end of L-BFGS": ("#1baf7a", "^")}
rows_v = []
with plt.style.context("default"):
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.8), sharey="row", gridspec_kw=dict(height_ratios=[3, 2]))
    for col, model_name in enumerate(["linear-loss", "learned-loss"]):
        h = hist[hist.model == model_name]
        adam, lbfgs = h[h.phase == "Adam"], h[h.phase == "L-BFGS"]
        layers = [name for name in LAYER_STYLE if h[name].notna().any()]

        # top: gradient during training (rolling median over 25 records within each phase, for readability)
        ax = axes[0, col]
        for name in layers:
            color, ls = LAYER_STYLE[name]
            ax.plot(adam.n_eval, adam[name].rolling(25, min_periods=1, center=True).median(), color=color, ls=ls, lw=1.3, label=name)
            ax.plot(lbfgs.n_eval, lbfgs[name].rolling(25, min_periods=1, center=True).median(), color=color, ls=ls, lw=1.3)
        ax.axvline(lbfgs.n_eval.min() - 1, color="0.45", lw=0.8)
        ax.text(lbfgs.n_eval.min() - 1, 0.03, " L-BFGS", transform=ax.get_xaxis_transform(), fontsize=8, color="0.3")
        ax.set_yscale("log")
        ax.set_xlabel("gradient evaluations")
        ax.set_title(f"{model_name} PINN ({DIAG_CURVE}, seed 0)", fontsize=10)
        ax.legend(fontsize=7.5, frameon=False, ncol=4, loc="upper right")
        ax.grid(alpha=0.3, lw=0.5)

        # bottom: gradient across the layers at three moments (median of the last 25 records at the two ends)
        ax = axes[1, col]
        groups = [[name for name in layers if name.startswith(prefix)] for prefix in "Tf"]
        positions = {name: i + 1 * name.startswith("f") for i, name in enumerate(layers)}
        for moment, part in {"start of Adam": adam.head(1), "end of Adam": adam.tail(25), "end of L-BFGS": lbfgs.tail(25)}.items():
            color, marker = MOMENT_STYLE[moment]
            size = part[layers].median()
            for g, group in enumerate(groups):
                if group:
                    ax.plot([positions[name] for name in group], size[group], color=color, marker=marker, ms=5, lw=1.3,
                            label=moment if g == 0 else None)
            rows_v.append(dict(model=model_name, moment=moment, **size.to_dict(), T1_over_T4=size["T1"] / size["T4"], T1_over_Tout=size["T1"] / size["Tout"]))
        ax.set_xticks(list(positions.values()))
        ax.set_xticklabels(layers)
        ax.set_yscale("log")
        ax.set_xlabel("layer (input → output)")
        ax.legend(fontsize=8, frameon=False)
        ax.grid(alpha=0.3, lw=0.5)
    axes[0, 0].set_ylim(top=axes[0, 0].get_ylim()[1] * 10)
    axes[0, 0].set_ylabel(r"gradient of the layer, $\|g\|/\sqrt{n}$")
    axes[1, 0].set_ylabel(r"gradient of the layer, $\|g\|/\sqrt{n}$")
    fig.suptitle("Gradient flow per layer (vanishing-gradient check)", fontsize=11)
    fig.tight_layout()
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/fig9_gradient_flow.jpg", dpi=150, bbox_inches="tight")
    plt.show()

print("Gradient of each layer at three moments, and the first layer against T4 and against the output layer")
print("A vanishing gradient would appear as a steady fall from T4 to T1 (T1_over_T4 far below 1); T1_over_Tout also contains the step at the output layer")
print(pd.DataFrame(rows_v).to_string(index=False, float_format=lambda x: f"{x:.3g}"))


### 12.3 The learned-loss PINN with the budget of the linear-loss PINN

The learned-loss PINN trains with 8000 Adam steps and at most 1000 L-BFGS iterations, the linear-loss PINN with 3000 and at most 300, although their parameter counts are nearly equal (Section 12.2). To see what the larger budget changes, the learned-loss PINN is trained here with 3000 + 300 on the data of Sections 11.1 and 11.2, with the same seeds, and compared with its 8000 + 1000 results, which are read from `results/loss_law_synthetic.csv` and `results/loss_law_lambda_sensitivity.csv`. `da_pct` is the error in $a$ against the true value on synthetic data and the change of $a$ against the single-exponential fit on the real curves. Everything else stays as it is, including the starting value of $\hat a$ (1 here, 0 in the linear-loss PINN). The table is saved in `results/same_budget_test.csv`.

**Result** (three seeds; `da_pct` in %, as mean and range; the two-valued columns give 3000 + 300 / 8000 + 1000).

| Case | `da_pct`, 3000 + 300 | `da_pct`, 8000 + 1000 | RMS of the network (°C) | Further check | Seeds stopped by the L-BFGS ceiling at 3000 + 300 |
|---|---|---|---|---|---|
| DS1 | +1.02 (+0.98 to +1.07) | +0.96 (+0.91 to +1.01) | 0.228 / 0.228 | physics loss 9.9e-6 / 1.1e-5 | 0 of 3 |
| DS2a | +2.38 (+1.67 to +3.42) | +5.81 (+5.17 to +6.32) | 0.376 / 0.286 | physics loss 2.0e-4 / 1.7e-5 | 3 of 3 |
| DS2b | +6.53 (+4.56 to +9.97) | +11.63 (+11.37 to +11.76) | 0.414 / 0.289 | physics loss 5.1e-4 / 1.7e-5 | 3 of 3 |
| DS3 | −0.61 (−0.69 to −0.56) | −0.75 (−0.94 to −0.53) | 0.484 / 0.459 | physics loss 8.6e-6 / 7.3e-6 | 1 of 3 |
| DS4 | +2.84 (+2.03 to +4.12) | +4.13 (+4.00 to +4.29) | 0.735 / 0.680 | physics loss 4.4e-5 / 2.7e-5 | 2 of 3 |
| power n=1.25, 600 s | −0.22 (−0.31 to −0.14) | −0.14 (−0.42 to +0.02) | 0.105 / 0.102 | error in $f$ 2.91 / 2.48 % | 0 of 3 |
| power n=1.25, 1200 s | +0.40 (+0.18 to +0.53) | +0.35 (+0.05 to +0.56) | 0.119 / 0.121 | error in $f$ 1.41 / 1.55 % | 0 of 3 |
| linear+quadratic, 600 s | −0.57 (−0.64 to −0.47) | −0.25 (−0.33 to −0.12) | 0.112 / 0.104 | error in $f$ 4.48 / 2.83 % | 0 of 3 |
| linear+quadratic, 1200 s | +0.38 (+0.32 to +0.42) | +0.34 (+0.29 to +0.43) | 0.128 / 0.112 | error in $f$ 1.90 / 1.46 % | 3 of 3 |

1. **The larger budget matters on the strongly nonlinear real curves.** With 3000 + 300 the change of $a$ is +2.4 % instead of +5.8 % on DS2a and +6.5 % instead of +11.6 % on DS2b, the spread over seeds is wider, the residual is higher, and the physics loss is 12 and 30 times larger. All seeds on these two curves stop at the L-BFGS ceiling, so the training has not finished. DS4 lies between (+2.8 % instead of +4.1 %).
2. **It matters little on the other cases.** On DS1, DS3 and the synthetic data the two budgets differ by at most 0.32 points in $a$. The largest error in $f$ on synthetic data rises to 4.48 % (from 2.83 %, mixed law, 600 s).
3. **Limits of this test.** Adam and L-BFGS are reduced together, so it does not show which of the two matters more. It also does not show that 8000 + 1000 is the smallest sufficient budget. A run with 3000 + 300 takes 9.7 s on average, against 27.7 s for the full budget on DS2b (Section 12.2).

In [ ]:
# Section 12.3: learned-loss PINN with 3000 Adam steps and at most 300 L-BFGS iterations
SAME_BUDGET = dict(adam_epochs=3000, lbfgs_iters=300)
rows_b = []
for law, f_true in LOSS_LAWS.items():
    for t_off_ in (600.0, 1200.0):
        for seed_ in range(3):
            t_, th_ = synthetic_case(law, t_off_, seed_)
            log = TrainLog(every=0)
            a_n, f_n, fit_n = train_loss_pinn(t_, th_, t_off_, seed=seed_, log=log, **SAME_BUDGET)
            grid_ = np.linspace(1, th_.max(), 30)
            rows_b.append(dict(budget="3000+300", set="synthetic", case=f"{law}, {t_off_:.0f} s", seed=seed_,
                               da_pct=100 * (a_n - a_true) / a_true, rms_network=np.sqrt(np.mean((fit_n - th_) ** 2)),
                               f_err_pct=100 * np.max(np.abs(f_n(grid_) - f_true(grid_))) / f_true(th_.max()), **log.cost()))
for label, ds in REAL_BY_LABEL.items():
    t_r, T_r = load_real(ds)
    th_r, t_off_r = T_r - ds["Tenv"], ds["t_off"]
    p_s, _ = single_tau_fit(t_r, th_r, t_off_r)
    for seed_ in range(3):
        log = TrainLog(every=0)
        a_n, f_n, fit_n, tm = train_loss_pinn(t_r, th_r, t_off_r, seed=seed_, log=log, return_terms=True, **SAME_BUDGET)
        rows_b.append(dict(budget="3000+300", set="real", case=label, seed=seed_, da_pct=100 * (a_n - p_s[0]) / p_s[0],
                           rms_network=np.sqrt(np.mean((fit_n - th_r) ** 2)), physics_loss=tm["physics"], **log.cost()))

syn = pd.read_csv("results/loss_law_synthetic.csv")
real = pd.read_csv("results/loss_law_lambda_sensitivity.csv").query("model == 'learned-loss' and physics_weight == 1.0")
reference = pd.concat([
    pd.DataFrame(dict(set="synthetic", case=syn.law + ", " + syn.t_off.astype(int).astype(str) + " s", seed=syn.seed,
                      da_pct=syn.loss_PINN, rms_network=syn.rms_PINN, f_err_pct=syn.f_err_pct)),
    pd.DataFrame(dict(set="real", case=real.dataset, seed=real.seed, da_pct=real.da_vs_single_pct,
                      rms_network=real.rms_network, physics_loss=real.physics_loss))]).assign(budget="8000+1000")
df_b = pd.concat([pd.DataFrame(rows_b), reference], ignore_index=True)
df_b.to_csv("results/same_budget_test.csv", index=False)
print(df_b.groupby(["set", "case", "budget"]).agg(
    da_mean=("da_pct", "mean"), da_min=("da_pct", "min"), da_max=("da_pct", "max"), rms_network=("rms_network", "mean"),
    f_err_pct=("f_err_pct", "mean"), physics_loss=("physics_loss", "mean"), lbfgs_iters=("lbfgs_iters", "mean"),
    lbfgs_evals=("lbfgs_evals", "mean")).to_string(float_format=lambda x: f"{x:.3g}"))


**Further budget checks.** The test above only lowers the budget of the learned-loss PINN. The next cell does two more things. It raises that budget to 16000 + 2000 on DS2a, DS2b and DS4, the curves where the budget matters, to see whether the 8000 + 1000 results are stable. It also trains the linear-loss PINN with 8000 + 1000 on the five real curves, so that both PINNs are compared with the same budget. The reference values with the usual budgets are read from `results/loss_law_lambda_sensitivity.csv`, and the table is saved in `results/budget_checks.csv`.

In [ ]:
# Section 12.3: a larger budget for the learned-loss PINN, the same budget for the linear-loss PINN
rows_c2 = []
for label, ds in REAL_BY_LABEL.items():
    t_r, T_r = load_real(ds)
    Tenv_r, t_off_r = ds["Tenv"], ds["t_off"]
    th_r = T_r - Tenv_r
    p_s, _ = single_tau_fit(t_r, th_r, t_off_r)
    for seed_ in range(3):
        log = TrainLog(every=0)
        with redirect_stdout(io.StringIO()):                                # hide the per-epoch log
            a_l, tau_l, T_lin = train_photothermal(t_r, T_r, t_off_r, Tenv_=Tenv_r, seed=seed_, adam_epochs=8000,
                                                   lbfgs_iters=1000, return_fit=True, log=log)
        rows_c2.append(dict(model="linear-loss", budget="8000+1000", case=label, seed=seed_, da_pct=100 * (a_l - p_s[0]) / p_s[0],
                            rms_network=np.sqrt(np.mean((T_lin - T_r) ** 2)), **log.cost()))
        if label in ("DS2a", "DS2b", "DS4"):
            log = TrainLog(every=0)
            a_n, f_n, fit_n, tm = train_loss_pinn(t_r, th_r, t_off_r, seed=seed_, adam_epochs=16000, lbfgs_iters=2000,
                                                  return_terms=True, log=log)
            rows_c2.append(dict(model="learned-loss", budget="16000+2000", case=label, seed=seed_, da_pct=100 * (a_n - p_s[0]) / p_s[0],
                                rms_network=np.sqrt(np.mean((fit_n - th_r) ** 2)), physics_loss=tm["physics"], **log.cost()))

ref_c2 = pd.read_csv("results/loss_law_lambda_sensitivity.csv").query("physics_weight == 1.0")
ref_c2 = pd.DataFrame(dict(model=ref_c2.model, budget=np.where(ref_c2.model == "linear-loss", "3000+300", "8000+1000"), case=ref_c2.dataset,
                           seed=ref_c2.seed, da_pct=ref_c2.da_vs_single_pct, rms_network=ref_c2.rms_network, physics_loss=ref_c2.physics_loss))
df_c2 = pd.concat([pd.DataFrame(rows_c2), ref_c2], ignore_index=True)
df_c2.to_csv("results/budget_checks.csv", index=False)
print(df_c2.groupby(["model", "case", "budget"]).agg(
    da_mean=("da_pct", "mean"), da_min=("da_pct", "min"), da_max=("da_pct", "max"), rms_network=("rms_network", "mean"),
    physics_loss=("physics_loss", "mean"), lbfgs_iters=("lbfgs_iters", "mean"), lbfgs_evals=("lbfgs_evals", "mean")).to_string(float_format=lambda x: f"{x:.3g}"))


### 12.4 Curve-fitting methods

Every non-neural estimator in this notebook is an ordinary, unweighted least-squares fit. They differ in the model, the data they use and the solver.

| Estimator | Function | Model fitted | Data used | Solver |
|---|---|---|---|---|
| Roper | `roper` | straight line through $\ln\theta$ against $t$, then $\hat a = \Delta T_{\max}/\hat\tau$ | cooling phase, while $\theta > 0.05\,\Delta T_{\max}$ | `numpy.polyfit`, degree 1 (linear least squares) |
| Whole-curve fit | `whole_curve_fit` | closed-form solution `heating_cooling`, free $a$ and $\tau$ | all points | `scipy.optimize.curve_fit`; with bounds it uses the Trust Region Reflective method (`trf`) |
| Single exponential | `single_tau_fit` | numerical ODE solution with $f = k\theta$, free $a$ and $k = 1/\tau$ | all points | `scipy.optimize.least_squares` (`trf`), with `solve_ivp` (RK45) for the ODE |
| Power law | `power_law_fit` | numerical ODE solution with $f = k\theta^n$, free $a$, $k$ and $n$ | all points | same as above, from five starting exponents, best fit kept |
| Recovery of $k$ (Section 12.5) | `recover_k` | $f = k\theta$, $f = k\theta^n$ and $f = k_1\theta + k_2\theta^2$, fitted to the learned $f(\theta)$ | 50 values of $\theta$ | closed form; `curve_fit` without bounds (Levenberg–Marquardt, `lm`); `numpy.linalg.lstsq` |

The Jacobians of `curve_fit` and `least_squares` come from finite differences. `whole_curve_fit` and `single_tau_fit` fit the same model by two routes, and their heating rates on the five real curves agree within 0.03 % (`results/real_data_results.csv` and `results/loss_law_real.csv`).

### 12.5 Can $k$ be recovered from the learned loss law?

The linear-loss PINN and `single_tau_fit` estimate two parameters, $a$ and $k = 1/\tau$, and `power_law_fit` estimates $a$, $k$ and $n$ of $f = k\theta^n$. The learned-loss PINN estimates $a$ directly, but its $f(\theta)$ is a network and contains no $k$. A $k$ can only be obtained by fitting a form to the learned $f$ after training, which `recover_k` does on 50 values of $\theta$ between $0.1\,\theta_{\max}$ and $\theta_{\max}$:

- $f = k\theta$ gives one number, equal to $1/\tau$ for a linear loss. For a nonlinear loss it is only an average, because $f/\theta$ changes with $\theta$.
- $f = k\theta^n$ gives $k$ and $n$. The unit of $k$ depends on $n$ (°C$^{1-n}$ s$^{-1}$), so the two must be read together.
- $f = k_1\theta + k_2\theta^2$ gives the two coefficients of the mixed law of Section 11.1.

On synthetic data the true values are known. Section 11.1 has no linear loss, so the first cell trains the learned-loss PINN on the two linear-loss experiments of Section 7 (three seeds each), where the true $k$ is $1/\tau = 1/300$ s$^{-1}$; it is the only training in this section. On the real curves there is no true value, so the recovered parameters are compared with the direct fits and, for $k = 1/\tau$, with the linear-loss PINN. The parameters of every run are saved in `results/recovered_k.csv`.

**Result on synthetic data** (mean over three seeds, error against the true value in brackets).

| Loss law | Heating | Parameter | True | Recovered from the PINN | Direct fit |
|---|---|---|---|---|---|
| linear | 1500 s | $k = 1/\tau$ | 3.333e-3 | 3.342e-3 (+0.26 %) | 3.338e-3 (+0.15 %) |
| linear | 300 s | $k = 1/\tau$ | 3.333e-3 | 3.312e-3 (−0.64 %) | 3.325e-3 (−0.26 %) |
| power law | 600 s | $k$ | 1.182e-3 | 1.138e-3 (−3.75 %) | 1.164e-3 (−1.53 %) |
| power law | 600 s | $n$ | 1.250 | 1.265 (+1.19 %) | 1.257 (+0.56 %) |
| power law | 1200 s | $k$ | 1.182e-3 | 1.245e-3 (+5.31 %) | 1.174e-3 (−0.66 %) |
| power law | 1200 s | $n$ | 1.250 | 1.234 (−1.27 %) | 1.252 (+0.17 %) |
| mixed law | 600 s | $k_1$ | 1.250e-3 | 1.194e-3 (−4.48 %) | — |
| mixed law | 600 s | $k_2$ | 6.250e-5 | 6.605e-5 (+5.68 %) | — |
| mixed law | 1200 s | $k_1$ | 1.250e-3 | 1.273e-3 (+1.83 %) | — |
| mixed law | 1200 s | $k_2$ | 6.250e-5 | 6.220e-5 (−0.48 %) | — |

**Result on the real curves** (mean over three seeds).

| Curve | $k = 1/\tau$, PINN | $k = 1/\tau$, single-exponential fit | $k = 1/\tau$, linear-loss PINN | $k$ of $k\theta^n$, PINN | $k$ of $k\theta^n$, power-law fit | $n$, PINN | $n$, power-law fit |
|---|---|---|---|---|---|---|---|
| DS1 | 2.652e-3 | 2.625e-3 | 2.620e-3 | 1.745e-3 | 1.695e-3 | 1.173 | 1.183 |
| DS2a | 4.481e-3 | 4.379e-3 | 4.350e-3 | 2.269e-3 | 1.878e-3 | 1.255 | 1.306 |
| DS2b | 4.592e-3 | 4.542e-3 | 4.523e-3 | 9.512e-4 | 7.121e-4 | 1.542 | 1.625 |
| DS3 | 2.627e-3 | 2.652e-3 | 2.650e-3 | 2.069e-3 | 2.532e-3 | 1.072 | 1.013 |
| DS4 | 4.628e-3 | 4.527e-3 | 4.511e-3 | 2.202e-3 | 2.324e-3 | 1.273 | 1.236 |

1. **On synthetic data $k$ can be recovered.** The error is at most 0.64 % for $k = 1/\tau$ of a linear loss, 5.31 % for $k$ and 1.27 % for $n$ of the power law, and 5.68 % for $k_1$ and $k_2$ of the mixed law. The heating rate $a$ of the same runs is within 0.35 %. The direct power-law fit, which is given the correct form, is closer (1.53 % for $k$, 0.56 % for $n$).
2. **On the real curves $k = 1/\tau$ is stable, $k$ of the power law is not.** $k = 1/\tau$ differs from the single-exponential fit by −0.95 % to +2.34 %, with a seed spread of at most 1.55 %. For the power law, $n$ differs from the direct fit by −0.083 to +0.059, and $k$ by −18.3 % to +33.6 %. The unit of $k$ depends on $n$, so a small change in $n$ moves $k$ strongly.
3. **For a nonlinear loss $k = 1/\tau$ is only an average** over the fitted range of $\theta$, as Section 12.6 shows.

In [ ]:
# Section 12.5: learned-loss PINN on linear-loss synthetic data, where k = 1/τ is known
f_rows = []
for exp_name, e in EXPERIMENTS.items():
    for seed_ in range(3):
        t_e, T_e = make_experiment(**e, seed=seed_)
        th_e = T_e - Tenv_true
        p_s, _ = single_tau_fit(t_e, th_e, e["t_off_e"])
        a_n, f_n, _ = train_loss_pinn(t_e, th_e, e["t_off_e"], seed=seed_)
        f_rows.append(learned_f_rows(exp_name, e["t_off_e"], seed_, th_e, a_n, f_n, p_s))
pd.concat(f_rows).to_csv("results/learned_f_linear.csv", index=False)


In [ ]:
# Section 12.5: k recovered from the learned f (no training)
def load_learned_f():
    return pd.concat([pd.read_csv("results/learned_f_real.csv").assign(group="real"),
                      pd.read_csv("results/learned_f_synthetic.csv").assign(group="nonlinear"),
                      pd.read_csv("results/learned_f_linear.csv").assign(group="linear")], ignore_index=True)


def recover_k(theta, f):
    # least-squares fits to the learned f(θ); θref as in power_law_fit, to decorrelate k and n
    k_linear = (theta @ f) / (theta @ theta)
    ref = theta.max() / 2
    (k_ref, n), _ = curve_fit(lambda x, k_, n_: k_ * ref * (x / ref) ** n_, theta, f, p0=(k_linear, 1.0))
    k1, k2 = np.linalg.lstsq(np.column_stack([theta, theta ** 2]), f, rcond=None)[0]
    return dict(k_linear=k_linear, k_power=k_ref * ref ** (1 - n), n=n, k1=k1, k2=k2)


rows_k = []
for (group, case, t_off_, seed_), d in load_learned_f().groupby(["group", "case", "t_off", "seed"], sort=False):
    first = d.iloc[0]
    rows_k.append(dict(group=group, case=case, t_off=t_off_, seed=seed_, a_PINN=first.a_PINN,
                       **recover_k(d.theta.values, d.f.values), a_single=first.a_single, k_single=first.k_single,
                       a_power=first.a_power, k_power_direct=first.k_power, n_direct=first.n_power))
rec = pd.DataFrame(rows_k)
rec.to_csv("results/recovered_k.csv", index=False)

# synthetic data: recovered parameters against the true values and against the direct fits
truth_all = {**TRUE_K, **{exp_name: {"k_linear": 1 / tau_true} for exp_name in EXPERIMENTS}}
direct_of = {"k_linear": "k_single", "k_power": "k_power_direct", "n": "n_direct"}
rows_t = []
for case, truth in truth_all.items():
    for t_off_, d in rec[rec.case == case].groupby("t_off"):
        a_direct = "a_single" if "k_linear" in truth else "a_power"
        for par, direct, true in [("a_PINN", a_direct, a_true)] + [(p, direct_of.get(p), v) for p, v in truth.items()]:
            rows_t.append(dict(case=case, t_off=t_off_, parameter=par, true=true, PINN=d[par].mean(), PINN_sd=d[par].std(ddof=1),
                               PINN_error_pct=100 * (d[par].mean() - true) / true,
                               direct_fit=d[direct].mean() if direct else np.nan,
                               direct_error_pct=100 * (d[direct].mean() - true) / true if direct else np.nan))
print("=== Synthetic data: mean over three seeds, k_linear = 1/τ ===")
print(pd.DataFrame(rows_t).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

# real curves: recovered parameters against the direct fits and the linear-loss PINN
r = rec[rec.group == "real"].groupby("case", sort=False)
linear_pinn_tau = pd.read_csv("results/real_data_results.csv").query("method == 'PINN + switch'").tau_est.values
real_table = pd.DataFrame(dict(a_PINN=r.a_PINN.mean(), k_PINN=r.k_linear.mean(), k_PINN_sd=r.k_linear.std(),
                               k_single_fit=r.k_single.mean(), k_linear_PINN=1 / linear_pinn_tau,
                               k_power_PINN=r.k_power.mean(), k_power_PINN_sd=r.k_power.std(), n_PINN=r.n.mean(), n_PINN_sd=r.n.std(),
                               k_power_fit=r.k_power_direct.mean(), n_fit=r.n_direct.mean()))
print("\n=== Real curves: mean over three seeds ===")
print(real_table.to_string(float_format=lambda x: f"{x:.4g}"))


### 12.6 Curves rebuilt from the recovered parameters

The ODE is solved with the parameters recovered from the learned-loss PINN, once with $(a, k)$ and once with $(a, k, n)$, and the result is compared with the data. On the real curves the figure also shows the linear-loss PINN with its own $a$ and $k = 1/\tau$ (seed 0, Section 10). The table also gives the RMS of the full learned $f$ (interpolated from the saved grid) and of the direct fits, which show how much is lost by reducing the network to parameters. The figure shows seed 0 and is saved in `results/plots/fig10_recovered_parameters.jpg`; the table in `results/recovered_parameters_rms.csv`.

**Result** (RMS against the data in °C, mean over three seeds).

| Curve | PINN $(a, k)$ | PINN $(a, k, n)$ | PINN, full learned $f$ | Single-exponential fit | Power-law fit |
|---|---|---|---|---|---|
| DS1 | 0.364 | 0.240 | 0.238 | 0.361 | 0.239 |
| DS2a | 0.941 | 0.494 | 0.300 | 0.783 | 0.430 |
| DS2b | 2.456 | 0.564 | 0.296 | 1.636 | 0.442 |
| DS3 | 0.574 | 0.614 | 0.469 | 0.568 | 0.566 |
| DS4 | 1.025 | 0.872 | 0.690 | 0.967 | 0.828 |
| power law, 600 s | 0.254 | 0.101 | 0.103 | 0.247 | 0.099 |
| power law, 1200 s | 0.551 | 0.123 | 0.121 | 0.531 | 0.099 |
| mixed law, 600 s | 0.318 | 0.100 | 0.104 | 0.309 | 0.100 |
| mixed law, 1200 s | 0.788 | 0.120 | 0.113 | 0.734 | 0.119 |
| linear, 1500 s | 0.099 | 0.102 | 0.103 | 0.099 | — |
| linear, 300 s | 0.093 | 0.092 | 0.092 | 0.092 | — |

1. **Linear-loss data.** All curves reach the noise level of 0.1 °C (0.092–0.103 °C), so here $(a, k)$ is enough.
2. **Nonlinear synthetic data.** $(a, k, n)$ stays close to the noise level (0.100–0.123 °C), while $(a, k)$ does not (0.254–0.788 °C).
3. **Real curves.** The linear-loss PINN with its own $a$ and $k$ gives 0.36, 0.78, 1.64, 0.57 and 0.97 °C on DS1, DS2a, DS2b, DS3 and DS4, the same as the single-exponential fit. The $(a, k)$ recovered from the learned-loss PINN is worse than the direct single-exponential fit on all five curves, by very little on DS1 and DS3 and most on DS2b (2.46 against 1.64 °C). The heating rate of the nonlinear law does not fit one average $k$: on DS2b the steady state $a/k$ is 27.9 °C, while the measured rise reaches 24.6 °C. $(a, k, n)$ is close to the direct power-law fit on DS1 (0.240 against 0.239 °C) and worse on the other four (0.494, 0.564, 0.614, 0.872 against 0.430, 0.442, 0.566, 0.828 °C). On DS3 it is also worse than $(a, k)$.
4. **The full learned $f$ has the lowest PINN residual** on the real curves (0.238–0.690 °C), so reducing the network to parameters loses information, most on DS2b (0.296 against 0.564 °C).

In [ ]:
# Section 12.6: curves rebuilt from the parameters recovered from the learned-loss PINN (no training)
def curve_data(group, case, t_off_, seed_):
    if group == "real":
        ds = REAL_BY_LABEL[case]
        t_r, T_r = load_real(ds)
        return t_r, T_r - ds["Tenv"]
    if group == "nonlinear":
        return synthetic_case(case, t_off_, seed_)
    t_e, T_e = make_experiment(**EXPERIMENTS[case], seed=seed_)
    return t_e, T_e - Tenv_true


rec = pd.read_csv("results/recovered_k.csv")
f_all = load_learned_f()
linear_pinn = pd.read_csv("results/real_data_results.csv").query("method == 'PINN + switch'")
linear_pinn = dict(zip(DS_LABELS, zip(linear_pinn.a_est, linear_pinn.tau_est)))      # seed 0, Section 10
rows_q, panels = [], []
for _, r in rec.iterrows():
    t_d, th_d = curve_data(r["group"], r.case, r.t_off, int(r.seed))
    rms = lambda y: np.sqrt(np.mean((y - th_d) ** 2))
    d = f_all[(f_all.case == r.case) & (f_all.t_off == r.t_off) & (f_all.seed == r.seed)]
    theta_f, f_f = np.r_[0.0, d.theta.values], np.r_[0.0, d.f.values]
    y_ak = simulate_loss(t_d, r.a_PINN, lambda x: r.k_linear * x, r.t_off)
    y_akn = simulate_loss(t_d, r.a_PINN, lambda x: r.k_power * x ** r.n, r.t_off)
    y_f = simulate_loss(t_d, r.a_PINN, lambda x: np.interp(x, theta_f, f_f), r.t_off)
    y_single = simulate_loss(t_d, r.a_single, lambda x: r.k_single * x, r.t_off)
    y_linear = heating_cooling(t_d, *linear_pinn[r.case], r.t_off, 0.0) if r["group"] == "real" else np.full(t_d.shape, np.nan)
    has_power = pd.notna(r.a_power)
    y_power = simulate_loss(t_d, r.a_power, lambda x: r.k_power_direct * x ** r.n_direct, r.t_off) if has_power else np.full(t_d.shape, np.nan)
    rows_q.append(dict(group=r["group"], case=r.case, t_off=r.t_off, seed=r.seed, rms_PINN_a_k=rms(y_ak), rms_PINN_a_k_n=rms(y_akn),
                       rms_PINN_learned_f=rms(y_f), rms_linear_PINN=rms(y_linear), rms_single_fit=rms(y_single), rms_power_fit=rms(y_power)))
    if r.seed == 0:
        title = r.case if r["group"] == "real" else f"{r.case}, {r.t_off:.0f} s" if r["group"] == "nonlinear" else f"linear loss, {r.case}"
        panels.append((title, t_d, th_d, r.t_off, y_ak, y_akn, rms(y_ak), rms(y_akn), y_linear, rms(y_linear)))

df_q = pd.DataFrame(rows_q)
df_q.to_csv("results/recovered_parameters_rms.csv", index=False)
print("RMS against the data in °C, mean over three seeds")
print(df_q.groupby(["group", "case", "t_off"], sort=False).mean(numeric_only=True).drop(columns="seed").round(3).to_string())

with plt.style.context("default"):
    fig, axes = plt.subplots(3, 4, figsize=(15.5, 9.2))
    for ax, (title, t_d, th_d, t_off_, y_ak, y_akn, rms_ak, rms_akn, y_linear, rms_linear) in zip(axes.ravel(), panels):
        ax.plot(t_d / 60, th_d, color="0.72", lw=3, label="data")
        ax.plot(t_d / 60, y_ak, color="#2a78d6", ls="--", lw=1.6, label=f"learned-loss PINN a, k (RMS {rms_ak:.2f} °C)")
        ax.plot(t_d / 60, y_akn, color="#eb6834", lw=1.6, label=f"learned-loss PINN a, k, n (RMS {rms_akn:.2f} °C)")
        if np.isfinite(y_linear).all():
            ax.plot(t_d / 60, y_linear, color="#1baf7a", ls=":", lw=1.8, label=f"linear-loss PINN a, k (RMS {rms_linear:.2f} °C)")
        ax.axvline(t_off_ / 60, color="0.45", lw=0.8)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("Time (min)", fontsize=8)
        ax.set_ylabel("θ (°C)", fontsize=8)
        ax.tick_params(labelsize=8)
        ax.legend(fontsize=7, frameon=False, loc="lower center")
        ax.grid(alpha=0.3, lw=0.5)
    for ax in axes.ravel()[len(panels):]:
        ax.set_visible(False)
    fig.suptitle("Curves rebuilt from the parameters recovered from the PINNs (seed 0)", fontsize=11)
    fig.tight_layout()
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/fig10_recovered_parameters.jpg", dpi=150, bbox_inches="tight")
    plt.show()


---
## 13 · Summary

| Finding | Value |
|---|---|
| Roper error at $t_{\mathrm{off}}/\tau = 0.5$ | −61.2 % |
| Roper error at $t_{\mathrm{off}}/\tau = 1$ | −36.5 % |
| Roper error at $t_{\mathrm{off}}/\tau = 2$ | −14.6 % |
| Roper error at $t_{\mathrm{off}}/\tau = 5$ | +0.8 % |
| Whole-curve fit mean error (synthetic sweep) | ≤ 0.2 % |
| PINN + switch mean error (synthetic sweep) | ≤ 0.2 % |
| PINN without switch input (five seeds) | −29.3 % to +10.0 % at $t_{\mathrm{off}}/\tau = 5$, −6.3 % to +0.14 % at $t_{\mathrm{off}}/\tau = 1$; with switch ≤ 0.7 % |
| Real data: Roper vs whole-curve fit (reference), difference in $a$ | −69.8 % to +1.3 % |
| Real data: PINN vs whole-curve fit | within 0.7 % in $a$, ≤ 0.6 % in $\tau$ |
| Real data: single-τ ODE misfit (whole-curve RMS) | 0.36–1.64 °C |
| Nonlinear loss (synthetic): Roper error in $a$ | −29 % to −37 % |
| Nonlinear loss (synthetic): learned-loss PINN, error in $a$ / in $f$ | ≤ 0.4 % / ≤ 2.9 % |
| Real data: residual of single-τ / power law / learned loss | 0.36–1.64 / 0.24–0.83 / 0.23–0.68 °C |
| Real data: learned-loss $a$ vs single-τ $a$ | −0.7 % to +11.6 % (seed spread ≤ 0.6 %) |
| Real data: DS4, learned-loss $a$ vs single-τ $a$ (15 runs) | +2.1 % to +14.0 % |
| Real data: learned-loss $a$ vs single-τ $a$, physics weight 0.1 / 1 / 10 (Section 11.7) | 0.1 and 1 differ by ≤ 2.7 points on all five curves; 10 removes the DS2b change (+11.6 % → −0.3 %) |
| Real data: held-out cooling tail, learned loss vs power law | better on DS2a, DS2b, DS4; worse on DS1, DS3 |
| Gradient flow per layer (DS2b, seed 0) | no vanishing gradient; `T1` / `T4` = 0.61–8.28 |
| Cost of one run (DS2b), linear-loss / learned-loss PINN | 30,703 / 31,183 parameters; 3345 / 8939 gradient evaluations; 10.1 / 27.7 s |
| Learned-loss PINN with 3000 + 300 instead of 8000 + 1000, real data, $a$ vs single-τ $a$ | DS2a +2.4 % vs +5.8 %; DS2b +6.5 % vs +11.6 %; DS4 +2.8 % vs +4.1 %; DS1 and DS3 almost unchanged |
| $k$ recovered from the learned $f$, synthetic data | linear loss ≤ 0.64 %; power law $k$ ≤ 5.31 %, $n$ ≤ 1.27 %; mixed law ≤ 5.68 % |
| $k = 1/\tau$ recovered from the learned $f$ vs single-τ fit, real data | −0.95 % to +2.34 % |
| Real data: residual of curves rebuilt from the PINN, $(a, k)$ / $(a, k, n)$ / full learned $f$ | 0.36–2.46 / 0.24–0.87 / 0.24–0.69 °C |

**Key takeaways:**

1. On synthetic data the Roper error is the heating-truncation factor $1 - e^{-t_{\mathrm{off}}/\tau}$ of Section 2: it is negative whenever the heating phase is shorter than ≈ 3τ (−61.2 % at $t_{\mathrm{off}}/\tau = 0.5$) and vanishes for long experiments.
2. Both the whole-curve least-squares fit and the PINN remove this bias by fitting the complete ODE solution to heating and cooling simultaneously (errors ≤ 0.2 %).
3. The switch input $[t,\,\mathrm{ReLU}(t-t_{\mathrm{off}})]$ is essential for the PINN: without it the error in $a$ over five seeds spans −29.3 % to +10.0 % (long) and −6.3 % to +0.14 % (short), with it every run stays within 0.7 %.
4. On the five real curves (Vikas 2023, Sánchez 2014, Armenta-Gamez 2026) the single-τ ODE fits only approximately (whole-curve RMS 0.36–1.64 °C). In four of the five curves the cooling phase alone gives a longer relaxation time than the complete trace (e.g. 709 s against 220 s for PEG-GNR), because the cooling tails stay above the initial baseline, so the Roper estimate of $a$ falls 19–70 % below the whole-curve value; only AuNR agrees (+1.3 %).
5. The linear-loss PINN agrees with the whole-curve fit to within 0.7 % in $a$ and 0.6 % in $\tau$ on all real curves, so it offers no accuracy advantage over least squares in the single-τ setting.
6. Section 11 shows where the PINN adds value: once the loss law is unknown, it learns $f(\theta)$ from the data, recovers $a$ within 0.4 % on synthetic data, and gives the lowest residual on all five real curves; out of sample its advantage holds only for the strongly nonlinear curves (Section 11.4), and the heating rate of DS4 is not well determined (Section 11.5). The real curves lose heat faster than linearly (power-law exponents 1.18–1.31 on three curves, close to natural convection), which also explains the large Roper errors on real data.